# Biohub A+B Tracking Ensemble + Combined V2/Model-C Primary Division Decoder

This hidden-test candidate preserves the validated `.921` A+B tracking
pipeline, learned motion correction, and registration-aware post-processing.
Model C is a separate division-only evidence branch:
it cannot alter fused detections, ordinary continuation probabilities, or the
initial ILP graph.

Model C runs its native training-time inference path (`0.99` detection,
`3.0 µm` pooled four-view TTA). Its compact parent/target competition evidence
is aligned to Division V2's broad `14/20 µm` candidates. A leakage-safe decoder,
trained on 175 videos and calibrated on a disjoint 20-video split, makes the
primary division decision at the frozen `0.96` parent threshold.

Every completed video first receives a valid A+B baseline shard. Legacy V2 does
not select or lock events first; it runs only if the combined decoder fails.
Any worker error, deadline, or failed fallback retains the valid A+B shard.


## Biohub Cell Tracking — True Multi-Model Ensemble

### Credits and original reference

This work used the following public resources from [@pilkwang](https://www.kaggle.com/pilkwang) as an initial reference:

- [Biohub Cell Tracking — Learned Graph with Gap Recovery](https://www.kaggle.com/code/pilkwang/biohub-cell-tracking-learned-graph-w-gap-recovery)
- [Biohub Tracking Support Pack — 50 Epoch v1](https://www.kaggle.com/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/versions/6)

The original notebook provided a useful reference for the learned TemporalUNet3D/transformer architecture, offline dependencies, ILP graph construction, and basic graph-recovery workflow. The original Model A checkpoint and its supporting artifacts come from the credited public support pack.

The original notebook was a **single-model pipeline**. It did not implement the true multi-model ensemble used in this submission.

Many thanks to pilkwang for publishing the original model, support pack, and reproducible baseline. Please visit and upvote the original work.


# Frozen `.921` A+B + Motion, with Combined V2/Native-C Division Selection


## Responsibilities and validation contract

- **Models A+B:** fused detections, continuation edges, and one shared ILP.
- **Motion corrector and post-processing:** unchanged from the `.921` notebook.
- **Legacy Division V2:** feature generator and failure fallback only.
- **Model C:** native detections and edge-competition evidence only.
- **Combined decoder:** 102 pair inputs and 144 source inputs, aligned by the
  exact frozen V2 feature contract, and responsible for the primary decision.
- **Selection:** fixed `0.96` threshold from the disjoint held-20 split, one
  event per linked lineage.
- **Conflict policy:** existing A+B/ILP forks and daughters are protected. The combined selector can
  replace ordinary continuation assignments only inside a complete accepted
  parent-to-two-daughter transaction.
- **Safety:** in-degree ≤1 and out-degree ≤2 are validated atomically. A worker
  failure leaves the already-written baseline shard intact.

Clean matched evidence favored Model C over V2 on unseen videos: grouped OOF
division Jaccard improved from `0.181` to `0.381`; on the disjoint held-20 split
it improved from `0.074` to `0.190`. These are selection diagnostics, not a
claimed hidden-test score. The Kaggle hidden result is the deployment test.

The four practice clips are not used to retune this notebook. The decoder
threshold remains frozen.


In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd

import signal

NOTEBOOK_WALL_START = time.monotonic()

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
_test_dir_override = os.environ.get("BIOHUB_TEST_DIR", "").strip()
TEST_DIR = Path(_test_dir_override) if _test_dir_override else COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"
REGISTRATION_DIAGNOSTICS_PATH = WORKING_DIR / "registration_diagnostics.csv"
DENSE_SHORT7_COMPONENTS_PATH = WORKING_DIR / "dense_short7_components.csv"
DENSE_SHORT7_NODES_PATH = WORKING_DIR / "dense_short7_nodes.csv"

METHOD = "unet_transformer"

# --- TRUE PRE-ILP ENSEMBLE (three slots; A+B enabled, C ready) ---
# Each enabled model sees the same images. Detection evidence is fused before
# peak extraction; edge evidence is fused on one shared graph before one ILP.
ENSEMBLE_MODELS = [
    {
        "name": "A",
        "enabled": True,
        "path": "/kaggle/input/datasets/tweakai/biohub-ensemble-2modelv3/weights/unet_transformer/split_0/edge_predictor_best.pth",
        "weight": float(os.environ.get("BIOHUB_MODEL_A_WEIGHT", "1.0")),
        "device": os.environ.get("BIOHUB_MODEL_A_DEVICE", "cuda:0"),
    },
    {
        "name": "B",
        "enabled": True,
        "path": "/kaggle/input/datasets/tweakai/biohub-ensemble-2modelv3/weights/unet_transformer/split_1/edge_predictor_best.pth",
        "weight": float(os.environ.get("BIOHUB_MODEL_B_WEIGHT", "1.0")),
        "device": os.environ.get("BIOHUB_MODEL_B_DEVICE", "cuda:1"),
    },
    {
        "name": "C",
        "enabled": False,  # C is never a fused ensemble member.
        "path": os.environ.get("BIOHUB_MODEL_C_PATH", "/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/weights/unet_transformer/split_0/edge_predictor_best.pth"),
        "weight": float(os.environ.get("BIOHUB_MODEL_C_WEIGHT", "0.5")),
        # Kaggle has two GPUs, so a future C defaults to sharing GPU 0.
        "device": os.environ.get("BIOHUB_MODEL_C_DEVICE", "cuda:0"),
    },
]
WEIGHTS_RELATIVE = ENSEMBLE_MODELS[0]["path"]

EXPERIMENT_TAG = "true_ensemble_907_learned_motion_cost"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"
DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
EDGE_THRESHOLD = float(os.environ.get("BIOHUB_EDGE_THRESHOLD", "0.54"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.5"))

# Empty for a real submission. Useful for local smoke tests, e.g. BIOHUB_SLICE=:1.
SLICE = os.environ.get("BIOHUB_SLICE", "").strip()

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"
RUN_VISUAL_EDA = os.environ.get("BIOHUB_RUN_VISUAL_EDA", "0") != "0"
RUN_DENSE_SHORT7_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_DENSE_SHORT7_DIAGNOSTICS", "1") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "9.5"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.52"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.78"))
MOTION_COST_CORRECTOR_ENABLED = os.environ.get("BIOHUB_MOTION_COST_CORRECTOR_ENABLED", "1") != "0"
MOTION_COST_CORRECTOR_PATH = Path(os.environ.get("BIOHUB_MOTION_COST_CORRECTOR_PATH", "/kaggle/input/datasets/tweakai/biohub-motion-corrector-v1/motion_corrector_best.pt"))
MOTION_COST_CORRECTOR_STRENGTH = float(os.environ.get("BIOHUB_MOTION_COST_CORRECTOR_STRENGTH", "1.0"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2800"))

# Confidence-gated global frame registration. Only abrupt, well-supported shifts
# are applied; low-confidence and ordinary-motion frames use the exact .906 path.
OUTPUT_REGISTRATION_AWARE = os.environ.get("BIOHUB_OUTPUT_REGISTRATION_AWARE", "1") != "0"
REGISTRATION_DOWNSAMPLE_Z = int(os.environ.get("BIOHUB_REGISTRATION_DOWNSAMPLE_Z", "1"))
REGISTRATION_DOWNSAMPLE_YX = int(os.environ.get("BIOHUB_REGISTRATION_DOWNSAMPLE_YX", "4"))
REGISTRATION_MIN_PSR = float(os.environ.get("BIOHUB_REGISTRATION_MIN_PSR", "8.0"))
REGISTRATION_MIN_APPLY_UM = float(os.environ.get("BIOHUB_REGISTRATION_MIN_APPLY_UM", "3.0"))
REGISTRATION_MAX_APPLY_UM = float(os.environ.get("BIOHUB_REGISTRATION_MAX_APPLY_UM", "40.0"))
REGISTRATION_LINEFIT_STABILIZE = os.environ.get("BIOHUB_REGISTRATION_LINEFIT_STABILIZE", "1") != "0"
REGISTRATION_GEOMETRY_VALIDATE = os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_VALIDATE", "1") != "0"
REGISTRATION_GEOMETRY_MIN_NODES = int(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_MIN_NODES", "20"))
REGISTRATION_GEOMETRY_DISTANCE_CAP_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_DISTANCE_CAP_UM", "8.0"))
REGISTRATION_GEOMETRY_COVERAGE_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_COVERAGE_UM", "4.0"))
REGISTRATION_GEOMETRY_VETO_MARGIN_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_VETO_MARGIN_UM", "0.35"))
REGISTRATION_GEOMETRY_VETO_COVERAGE_DROP = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_VETO_COVERAGE_DROP", "0.03"))
REGISTRATION_GEOMETRY_RESCUE_MIN_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_RESCUE_MIN_UM", "2.0"))
REGISTRATION_GEOMETRY_RESCUE_MARGIN_UM = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_RESCUE_MARGIN_UM", "0.25"))
REGISTRATION_GEOMETRY_RESCUE_COVERAGE_GAIN = float(os.environ.get("BIOHUB_REGISTRATION_GEOMETRY_RESCUE_COVERAGE_GAIN", "0.04"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "5.4"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.052"))#.52
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2200"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.1"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "0") != "1"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "7"))
OUTPUT_ADAPTIVE_MIN_TRACK_LEN = os.environ.get("BIOHUB_OUTPUT_ADAPTIVE_MIN_TRACK_LEN", "1") != "0"
OUTPUT_LOW_DENSITY_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_LOW_DENSITY_MIN_TRACK_LEN", "9"))
OUTPUT_DENSE_NODES_PER_FRAME = float(os.environ.get("BIOHUB_OUTPUT_DENSE_NODES_PER_FRAME", "500.0"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.78"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "3"))
GAP_REFINE_BG_PERCENTILE = float(os.environ.get("BIOHUB_GAP_REFINE_BG_PERCENTILE", "55.0"))
GAP_REFINE_CENTER_SIGMA_UM = float(os.environ.get("BIOHUB_GAP_REFINE_CENTER_SIGMA_UM", "1.60"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "1"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "9.2"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "3.9"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0032"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "140"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.0045"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "6.85"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.45"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.0072"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.00375"))


CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "true_ensemble_models": [{k: v for k, v in m.items() if k != "path"} | {"path": m["path"]} for m in ENSEMBLE_MODELS],
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "edge_threshold": EDGE_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "run_visual_eda": RUN_VISUAL_EDA,
    "run_dense_short7_diagnostics": RUN_DENSE_SHORT7_DIAGNOSTICS,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_cost_corrector_enabled": MOTION_COST_CORRECTOR_ENABLED,
    "motion_cost_corrector_path": str(MOTION_COST_CORRECTOR_PATH),
    "motion_cost_corrector_strength": MOTION_COST_CORRECTOR_STRENGTH,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_registration_aware": OUTPUT_REGISTRATION_AWARE,
    "registration_downsample_z": REGISTRATION_DOWNSAMPLE_Z,
    "registration_downsample_yx": REGISTRATION_DOWNSAMPLE_YX,
    "registration_min_psr": REGISTRATION_MIN_PSR,
    "registration_min_apply_um": REGISTRATION_MIN_APPLY_UM,
    "registration_max_apply_um": REGISTRATION_MAX_APPLY_UM,
    "registration_linefit_stabilize": REGISTRATION_LINEFIT_STABILIZE,
    "registration_geometry_validate": REGISTRATION_GEOMETRY_VALIDATE,
    "registration_geometry_min_nodes": REGISTRATION_GEOMETRY_MIN_NODES,
    "registration_geometry_distance_cap_um": REGISTRATION_GEOMETRY_DISTANCE_CAP_UM,
    "registration_geometry_coverage_um": REGISTRATION_GEOMETRY_COVERAGE_UM,
    "registration_geometry_veto_margin_um": REGISTRATION_GEOMETRY_VETO_MARGIN_UM,
    "registration_geometry_veto_coverage_drop": REGISTRATION_GEOMETRY_VETO_COVERAGE_DROP,
    "registration_geometry_rescue_min_um": REGISTRATION_GEOMETRY_RESCUE_MIN_UM,
    "registration_geometry_rescue_margin_um": REGISTRATION_GEOMETRY_RESCUE_MARGIN_UM,
    "registration_geometry_rescue_coverage_gain": REGISTRATION_GEOMETRY_RESCUE_COVERAGE_GAIN,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_adaptive_min_track_len": OUTPUT_ADAPTIVE_MIN_TRACK_LEN,
    "output_low_density_min_track_len": OUTPUT_LOW_DENSITY_MIN_TRACK_LEN,
    "output_dense_nodes_per_frame": OUTPUT_DENSE_NODES_PER_FRAME,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
}


print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

In [ ]:
def _read_json(path: Path) -> dict:
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}


def _scale_from_root_meta(root_meta: dict) -> tuple[float, float, float]:
    try:
        transforms = root_meta["attributes"]["multiscales"][0]["datasets"][0]["coordinateTransformations"]
        for transform in transforms:
            if transform.get("type") == "scale":
                scale = transform.get("scale", [])
                if len(scale) >= 4:
                    return tuple(float(v) for v in scale[-3:])
    except Exception:
        pass
    return (1.625, 0.40625, 0.40625)


def _quantiles_from_root_meta(root_meta: dict) -> dict:
    return root_meta.get("attributes", {}).get("image_statistics", {}).get("quantiles", {}) or {}


def _estimated_nodes_from_geff(geff_path: Path) -> int | None:
    meta = _read_json(geff_path / "zarr.json")
    value = (
        meta.get("attributes", {})
        .get("geff", {})
        .get("extra", {})
        .get("estimated_number_of_nodes")
    )
    return int(value) if value is not None else None


def _metadata_rows(split_name: str, split_dir: Path) -> list[dict[str, object]]:
    if not split_dir.exists():
        return []
    rows = []
    for zarr_path in sorted(split_dir.glob("*.zarr")):
        stem = zarr_path.name[:-5]
        arr_meta = _read_json(zarr_path / "0" / "zarr.json")
        root_meta = _read_json(zarr_path / "zarr.json")
        shape = arr_meta.get("shape", [None, None, None, None])
        chunks = arr_meta.get("chunk_grid", {}).get("configuration", {}).get("chunk_shape", [])
        scale = _scale_from_root_meta(root_meta)
        quantiles = _quantiles_from_root_meta(root_meta)
        rows.append({
            "split": split_name,
            "dataset": stem,
            "embryo": stem.split("_")[0],
            "T": shape[0] if len(shape) > 0 else None,
            "Z": shape[1] if len(shape) > 1 else None,
            "Y": shape[2] if len(shape) > 2 else None,
            "X": shape[3] if len(shape) > 3 else None,
            "dtype": arr_meta.get("data_type"),
            "chunk_shape": tuple(chunks) if chunks else None,
            "scale_z_um": scale[0],
            "scale_y_um": scale[1],
            "scale_x_um": scale[2],
            "q001": quantiles.get("0.001"),
            "q999": quantiles.get("0.999"),
            "estimated_nodes": _estimated_nodes_from_geff(split_dir / f"{stem}.geff"),
        })
    return rows


metadata_rows = []
metadata_rows.extend(_metadata_rows("train", COMP_DIR / "train"))
metadata_rows.extend(_metadata_rows("test", TEST_DIR))
meta_df = pd.DataFrame(metadata_rows)

if len(meta_df):
    display(
        meta_df.groupby(["split", "T", "Z", "Y", "X", "dtype", "chunk_shape", "scale_z_um", "scale_y_um", "scale_x_um"], dropna=False)
        .size()
        .reset_index(name="videos")
    )

    split_summary = meta_df.groupby("split").agg(
        videos=("dataset", "count"),
        embryos=("embryo", "nunique"),
        estimated_nodes_median=("estimated_nodes", "median"),
        estimated_nodes_max=("estimated_nodes", "max"),
        q001_median=("q001", "median"),
        q999_median=("q999", "median"),
    ).reset_index()
    display(split_summary)

    gate_rows = []
    for radius_um in [2.65, 7.0, OUTPUT_EDGE_MAX_UM]:
        gate_rows.append({
            "radius_um": radius_um,
            "z_voxels": radius_um / 1.625,
            "y_voxels": radius_um / 0.40625,
            "x_voxels": radius_um / 0.40625,
        })
    display(pd.DataFrame(gate_rows).round(3))

    if meta_df["estimated_nodes"].notna().any():
        import matplotlib.pyplot as plt
        train_est = meta_df.loc[meta_df["estimated_nodes"].notna(), "estimated_nodes"].astype(float)
        fig, axes = plt.subplots(1, 2, figsize=(12.0, 3.6), constrained_layout=True)
        axes[0].hist(train_est, bins=24)
        axes[0].set_title("Estimated cells per train video")
        axes[0].set_xlabel("estimated nodes")
        axes[0].set_ylabel("videos")
        axes[1].scatter(meta_df.index, meta_df["estimated_nodes"], s=20, alpha=0.65)
        axes[1].set_title("Estimated-node density by sample")
        axes[1].set_xlabel("sample index")
        axes[1].set_ylabel("estimated nodes")
        plt.show()
else:
    print("No Zarr metadata found. This is expected only outside the Kaggle data mount.")


In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)


In [ ]:
import numpy as np


def _read_one_frame_for_visual(zarr_path: Path, t: int) -> np.ndarray:
    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())
    shape = tuple(int(v) for v in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    frame_shape = shape[1:]
    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    try:
        import blosc2 as _blosc2
        raw = chunk_path.read_bytes()
        arr = np.frombuffer(_blosc2.decompress(raw), dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            return arr.reshape(frame_shape).copy()
    except Exception:
        pass
    import zarr
    return np.asarray(zarr.open(zarr_path / "0", mode="r")[t])


if RUN_VISUAL_EDA:
    try:
        import matplotlib.pyplot as plt

        visual_candidates = sorted(TEST_DIR.glob("*.zarr")) if TEST_DIR.exists() else []
        if not visual_candidates:
            print("Visual EDA skipped: no test volumes were found.")
        else:
            visual_path = visual_candidates[0]
            meta = json.loads((visual_path / "0" / "zarr.json").read_text())
            shape = tuple(int(v) for v in meta["shape"])
            t_vis = min(max(shape[0] // 2, 0), shape[0] - 1)
            frame = _read_one_frame_for_visual(visual_path, t_vis)
            clip_hi = float(np.percentile(frame, 99.7))
            clip_lo = float(np.percentile(frame, 1.0))
            z_proj = np.clip(frame.max(axis=0), clip_lo, clip_hi)
            y_proj = np.clip(frame.max(axis=1), clip_lo, clip_hi)
            x_proj = np.clip(frame.max(axis=2), clip_lo, clip_hi)

            fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.5), constrained_layout=True)
            axes[0, 0].imshow(z_proj, cmap="magma")
            axes[0, 0].set_title(f"Z projection: {visual_path.stem}, t={t_vis}")
            axes[0, 0].set_xlabel("x voxel")
            axes[0, 0].set_ylabel("y voxel")

            axes[0, 1].imshow(y_proj, cmap="magma", aspect="auto")
            axes[0, 1].set_title("Y projection")
            axes[0, 1].set_xlabel("x voxel")
            axes[0, 1].set_ylabel("z voxel")

            axes[1, 0].imshow(x_proj, cmap="magma", aspect="auto")
            axes[1, 0].set_title("X projection")
            axes[1, 0].set_xlabel("y voxel")
            axes[1, 0].set_ylabel("z voxel")

            sample = frame.ravel()[::max(1, frame.size // 250_000)]
            axes[1, 1].hist(sample, bins=80, log=True)
            axes[1, 1].axvline(clip_lo, color="black", linewidth=1, alpha=0.6)
            axes[1, 1].axvline(clip_hi, color="black", linewidth=1, alpha=0.6)
            axes[1, 1].set_title("Intensity distribution")
            axes[1, 1].set_xlabel("uint16 intensity")
            axes[1, 1].set_ylabel("voxels, log scale")
            plt.show()
    except Exception as exc:
        print(f"Visual EDA skipped safely: {type(exc).__name__}: {exc}")


In [ ]:
# Frozen A+B graph plus native Model-C division evidence.
MODEL_C_PRIMARY_ARTIFACT_DIR = Path(os.environ.get(
    "BIOHUB_MODEL_C_PRIMARY_ARTIFACT_DIR",
    "/kaggle/input/datasets/tweakai/biohub-model-c-combined-primary-v1",
))
MODEL_C_CHECKPOINT = MODEL_C_PRIMARY_ARTIFACT_DIR / "model_c" / "edge_predictor_best.pth"
MODEL_C_EVIDENCE_DIR = WORKING_DIR / "model_c_native_evidence"
MODEL_C_PRIMARY_THRESHOLD = float(os.environ.get("BIOHUB_MODEL_C_PRIMARY_THRESHOLD", "0.96"))
if not MODEL_C_CHECKPOINT.exists():
    raise FileNotFoundError(f"Safe Model C checkpoint not found: {MODEL_C_CHECKPOINT}")

def list_test_stems() -> list[str]:
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory does not exist: {TEST_DIR}")
    stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr"))
    if not stems:
        raise FileNotFoundError(f"No test .zarr files found in {TEST_DIR}")
    return stems


test_stems = list_test_stems()
print(f"Found {len(test_stems)} test videos")

enabled_models = [m for m in ENSEMBLE_MODELS if m["enabled"] and m["weight"] > 0]
if not enabled_models:
    raise RuntimeError("No ensemble models enabled")
for member in enabled_models:
    if not member["path"] or not Path(member["path"]).exists():
        raise FileNotFoundError(f"{member['name']} checkpoint not found: {member['path']}")

# Materialize the custom predictor inside the extracted support repo.
TRUE_ENSEMBLE_SCRIPT = '#!/usr/bin/env python3\n"""True pre-ILP ensemble for Biohub UNet + node-transformer models.\n\nDetection logits are averaged before local-max extraction. Every member then\nscores the same detections and candidate matrix; edge probabilities are\naveraged before one ILP solve. Supports 1-3 weighted members on configurable\nCUDA devices.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport json\nimport os\nimport shutil\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport polars as pl\nimport torch\nimport torch.nn.functional as F\nimport zarr\nfrom tqdm import tqdm\nimport tracksdata as td\n\nsys.path.insert(0, str(Path(__file__).parent))\nimport predict_unet_transformer as base\nfrom predict_unet_transformer import PredictConfig\nfrom train_unet_transformer import extract_pos_features, _POS_EMBED_DIM, DEFAULT_METHOD\nfrom biohub_tracking.io import open_dataset, save_graph\nfrom dataspec import USERNAME\n\n\n@dataclass\nclass Member:\n    name: str\n    path: Path\n    weight: float\n    det_logit_bias: float\n    device: torch.device\n    model: object\n    window_size: int\n    downsample: tuple[int, ...]\n\n\ndef load_members(specs: list[tuple[str, Path, float, float, str]]) -> list[Member]:\n    members = []\n    for name, path, weight, det_logit_bias, device_name in specs:\n        if weight <= 0:\n            continue\n        if not path.exists():\n            raise FileNotFoundError(f"{name} checkpoint not found: {path}")\n        device = torch.device(device_name if torch.cuda.is_available() else "cpu")\n        model, window_size, downsample = base.load_model(path, device)\n        members.append(Member(name, path, weight, det_logit_bias, device, model, window_size, downsample))\n        print(\n            f"Loaded {name}: weight={weight:g} det_logit_bias={det_logit_bias:+.6f} "\n            f"device={device} path={path}", flush=True,\n        )\n    if not members:\n        raise ValueError("No enabled ensemble members")\n    ref = (members[0].window_size, members[0].downsample)\n    for member in members[1:]:\n        if (member.window_size, member.downsample) != ref:\n            raise ValueError(\n                "All members must share window_size/downsample; "\n                f"{members[0].name}={ref}, {member.name}="\n                f"{(member.window_size, member.downsample)}"\n            )\n    return members\n\n\ndef weighted_average(tensors: list[torch.Tensor], weights: list[float], device: torch.device):\n    total = float(sum(weights))\n    out = tensors[0].to(device, non_blocking=True) * (weights[0] / total)\n    for tensor, weight in zip(tensors[1:], weights[1:]):\n        out = out + tensor.to(device, non_blocking=True) * (weight / total)\n    return out\n\n\ndef top_other_parent(probability: torch.Tensor):\n    if probability.shape[0] == 1:\n        best = probability[0]\n        row = torch.zeros_like(best, dtype=torch.long)\n        return best, row, torch.zeros_like(best)\n    values, rows = torch.topk(probability, 2, dim=0, sorted=True)\n    return values[0], rows[0], values[1]\n\n\n@torch.no_grad()\ndef predict_video_true_ensemble(\n    members: list[Member], ds_path: Path, cfg: PredictConfig,\n    max_frames: int | None = None,\n    division_member: Member | None = None,\n    division_pool_um: float = 3.0,\n    division_radius_um: float = 20.0,\n    division_topk: int = 16,\n    division_min_probability: float = 0.01,\n):\n    primary = members[0].device\n    W = members[0].window_size\n    downsample = members[0].downsample\n    weights = [member.weight for member in members]\n    if division_member is not None:\n        if division_member.window_size != 2:\n            raise ValueError(f"Model C must use window=2, got {division_member.window_size}")\n        if tuple(division_member.downsample) != tuple(downsample):\n            raise ValueError(\n                f"Model C downsample {division_member.downsample} != A+B {downsample}"\n            )\n\n    ds = open_dataset(ds_path, normalize=False, load_image=False, downsample=downsample)\n    if "0.001" not in ds.quantiles or "0.999" not in ds.quantiles:\n        raise ValueError(f"Zarr attrs missing image_statistics.quantiles for {ds_path}")\n    zarr_arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]\n    q_low, q_high = float(ds.quantiles["0.001"]), float(ds.quantiles["0.999"])\n    T = ds.image_shape[0] if max_frames is None else min(ds.image_shape[0], max_frames)\n    image_shape = (T,) + ds.image_shape[1:]\n    target_shape = list(image_shape[1:])\n    ds_arr = np.asarray(downsample, dtype=np.float32)\n    all_runtime_members = members + ([division_member] if division_member is not None else [])\n    ds_arr_by_device = {\n        str(m.device): torch.as_tensor(ds_arr, device=m.device) for m in all_runtime_members\n    }\n    voxel_size = tuple(s * d for s, d in zip(ds.scale, downsample))\n    pool_k = base.pool_kernel_from_um(cfg.pool_kernel_um, voxel_size)\n    division_pool_k = base.pool_kernel_from_um(division_pool_um, voxel_size)\n    original_spacing = np.asarray(ds.scale, dtype=np.float32)\n\n    seen_frames, seen_pairs = set(), set()\n    coord_lists, coord_offset, all_edges = [], {}, []\n    division_seen_frames, division_seen_pairs = set(), set()\n    division_coord_by_t, division_offset = {}, {}\n    division_node_count = 0\n    division_evidence = {key: [] for key in (\n        "source_id", "target_id", "probability",\n        "alternative_parent_probability", "is_target_winner",\n        "distance_um", "source_target_rank",\n    )}\n    global_node_count = 0\n    stride = max(W - 1, 1)\n    starts = list(range(0, T - W + 1, stride))\n    if not starts or starts[-1] + W < T:\n        last = max(T - W, 0)\n        if not starts or last != starts[-1]:\n            starts.append(last)\n\n    for ws in tqdm(starts, desc="  true-ensemble windows", leave=False):\n        frame_indices = list(range(ws, ws + W))\n        imgs_cpu = torch.stack([\n            base._load_frame(zarr_arr, t, target_shape, downsample) for t in frame_indices\n        ])\n        imgs_cpu = ((imgs_cpu - q_low) / (q_high - q_low + 1e-6)).clamp(0.0).unsqueeze(0)\n\n        encoded = []\n        for member in members:\n            imgs = imgs_cpu.to(member.device, non_blocking=True)\n            feats, det = member.model.encode(imgs)\n            if cfg.det_tta:\n                for dims in [(-1,), (-2,), (-2, -1)]:\n                    _, det_flip = member.model.encode(imgs.flip(dims))\n                    for f in range(W):\n                        det[f] = det[f] + det_flip[f].flip(dims)\n                det = [value / 4.0 for value in det]\n            encoded.append((feats, det))\n\n        division_features = division_detection = None\n        if division_member is not None:\n            division_images = imgs_cpu.to(division_member.device, non_blocking=True)\n            division_features, division_detection = division_member.model.encode(division_images)\n            if cfg.det_tta:\n                for dims in [(-1,), (-2,), (-2, -1)]:\n                    _, flipped_detection = division_member.model.encode(\n                        division_images.flip(dims)\n                    )\n                    for local_frame in range(2):\n                        division_detection[local_frame] = (\n                            division_detection[local_frame]\n                            + flipped_detection[local_frame].flip(dims)\n                        )\n                division_detection = [value / 4.0 for value in division_detection]\n\n        fused_det = [\n            weighted_average([\n                encoded[k][1][f] + members[k].det_logit_bias\n                for k in range(len(members))\n            ], weights, primary)\n            for f in range(W)\n        ]\n        for f_idx, t in enumerate(frame_indices):\n            if t in seen_frames:\n                continue\n            arr = base._detect_cells_pooled(fused_det[f_idx][0], t, cfg.det_threshold, pool_k)\n            coord_offset[t] = (global_node_count, global_node_count + len(arr))\n            global_node_count += len(arr)\n            coord_lists.append(arr)\n            seen_frames.add(t)\n        if division_member is not None:\n            for f_idx, t in enumerate(frame_indices):\n                if t in division_seen_frames:\n                    continue\n                native = base._detect_cells_pooled(\n                    division_detection[f_idx][0], t, cfg.det_threshold, division_pool_k\n                ).astype(np.int32)\n                native[:, 1:] *= np.asarray(downsample, np.int32)\n                division_coord_by_t[t] = native\n                division_offset[t] = (\n                    division_node_count, division_node_count + len(native)\n                )\n                division_node_count += len(native)\n                division_seen_frames.add(t)\n\n        coords_so_far = np.concatenate(coord_lists) if coord_lists else np.empty((0, 4), np.int16)\n        for f_idx in range(W - 1):\n            t_src, t_tgt = frame_indices[f_idx], frame_indices[f_idx + 1]\n            if (t_src, t_tgt) in seen_pairs:\n                continue\n            seen_pairs.add((t_src, t_tgt))\n\n            # Exact native Model-C evidence used by decoder training:\n            # 0.99 detection, 3um pooled TTA, source-axis competition,\n            # 20um radius, top-16 targets, minimum probability 0.01.\n            if (\n                division_member is not None\n                and (t_src, t_tgt) not in division_seen_pairs\n            ):\n                division_seen_pairs.add((t_src, t_tgt))\n                native_src = division_coord_by_t.get(\n                    t_src, np.empty((0, 4), np.int32)\n                )\n                native_tgt = division_coord_by_t.get(\n                    t_tgt, np.empty((0, 4), np.int32)\n                )\n                if len(native_src) and len(native_tgt):\n                    dev = division_member.device\n                    source_raw = native_src[:, 1:].astype(np.float32)\n                    target_raw = native_tgt[:, 1:].astype(np.float32)\n                    source_ds = source_raw / ds_arr\n                    target_ds = target_raw / ds_arr\n                    n_native_source, n_native_target = len(source_ds), len(target_ds)\n                    src_tensor = torch.from_numpy(source_ds).unsqueeze(0).to(dev)\n                    tgt_tensor = torch.from_numpy(target_ds).unsqueeze(0).to(dev)\n                    source_rel = np.column_stack([\n                        np.zeros(n_native_source, np.float32), source_ds\n                    ])\n                    target_rel = np.column_stack([\n                        np.ones(n_native_target, np.float32), target_ds\n                    ])\n                    native_window_shape = (2,) + tuple(int(x) for x in ds.image_shape[1:])\n                    pos_src = torch.from_numpy(\n                        extract_pos_features(source_rel, native_window_shape).astype(np.float32)\n                    ).unsqueeze(0).to(dev)\n                    pos_tgt = torch.from_numpy(\n                        extract_pos_features(target_rel, native_window_shape).astype(np.float32)\n                    ).unsqueeze(0).to(dev)\n                    mask_src = torch.ones(\n                        1, n_native_source, dtype=torch.bool, device=dev\n                    )\n                    mask_tgt = torch.ones(\n                        1, n_native_target, dtype=torch.bool, device=dev\n                    )\n                    feat_src = division_member.model._index_features(\n                        division_features[:, f_idx], src_tensor, mask_src\n                    )\n                    feat_tgt = division_member.model._index_features(\n                        division_features[:, f_idx + 1], tgt_tensor, mask_tgt\n                    )\n                    logits = division_member.model.predict_edges(\n                        feat_src, feat_tgt,\n                        src_tensor * ds_arr_by_device[str(dev)],\n                        tgt_tensor * ds_arr_by_device[str(dev)],\n                        pos_src, pos_tgt, mask_src, mask_tgt,\n                    )[0]\n                    probability = torch.softmax(logits, dim=0).float()\n                    source_um = torch.as_tensor(\n                        source_raw * original_spacing, device=dev\n                    ).float()\n                    target_um = torch.as_tensor(\n                        target_raw * original_spacing, device=dev\n                    ).float()\n                    distance = torch.cdist(source_um, target_um)\n                    eligible = distance <= division_radius_um\n                    masked = probability.masked_fill(~eligible, -1.0)\n                    k = min(division_topk, n_native_target)\n                    values, targets = torch.topk(masked, k, dim=1, sorted=True)\n                    keep = values >= division_min_probability\n                    best_prob, best_row, second_prob = top_other_parent(probability)\n                    source_rows = torch.arange(\n                        n_native_source, device=dev\n                    ).unsqueeze(1).expand_as(targets)\n                    picked_source = source_rows[keep]\n                    picked_target = targets[keep]\n                    target_best = best_row[picked_target]\n                    alternative = torch.where(\n                        target_best == picked_source,\n                        second_prob[picked_target],\n                        best_prob[picked_target],\n                    )\n                    source_start = division_offset[t_src][0]\n                    target_start = division_offset[t_tgt][0]\n                    division_evidence["source_id"].append(\n                        (picked_source + source_start).cpu().numpy().astype(np.int64)\n                    )\n                    division_evidence["target_id"].append(\n                        (picked_target + target_start).cpu().numpy().astype(np.int64)\n                    )\n                    division_evidence["probability"].append(\n                        values[keep].cpu().numpy().astype(np.float32)\n                    )\n                    division_evidence["alternative_parent_probability"].append(\n                        alternative.cpu().numpy().astype(np.float32)\n                    )\n                    division_evidence["is_target_winner"].append(\n                        (target_best == picked_source).cpu().numpy().astype(np.uint8)\n                    )\n                    division_evidence["distance_um"].append(\n                        distance[picked_source, picked_target].cpu().numpy().astype(np.float32)\n                    )\n                    rank_grid = torch.arange(\n                        k, device=dev\n                    ).unsqueeze(0).expand_as(targets)\n                    division_evidence["source_target_rank"].append(\n                        rank_grid[keep].cpu().numpy().astype(np.int16)\n                    )\n\n            s_src, e_src = coord_offset.get(t_src, (0, 0))\n            s_tgt, e_tgt = coord_offset.get(t_tgt, (0, 0))\n            if e_src == s_src or e_tgt == s_tgt:\n                continue\n            c_src, c_tgt = coords_so_far[s_src:e_src], coords_so_far[s_tgt:e_tgt]\n            n_src, n_tgt = len(c_src), len(c_tgt)\n            c_src_rel, c_tgt_rel = c_src.copy(), c_tgt.copy()\n            c_src_rel[:, 0], c_tgt_rel[:, 0] = f_idx, f_idx + 1\n            window_shape = (W,) + image_shape[1:]\n\n            member_logits = []\n            for member, (features, _) in zip(members, encoded):\n                dev = member.device\n                src = torch.from_numpy(c_src[:, 1:].astype(np.float32)).unsqueeze(0).to(dev)\n                tgt = torch.from_numpy(c_tgt[:, 1:].astype(np.float32)).unsqueeze(0).to(dev)\n                pos_src = torch.from_numpy(extract_pos_features(c_src_rel, window_shape)).unsqueeze(0).to(dev)\n                pos_tgt = torch.from_numpy(extract_pos_features(c_tgt_rel, window_shape)).unsqueeze(0).to(dev)\n                mask_src = torch.ones(1, n_src, dtype=torch.bool, device=dev)\n                mask_tgt = torch.ones(1, n_tgt, dtype=torch.bool, device=dev)\n                feat_src = member.model._index_features(features[:, f_idx], src, mask_src)\n                feat_tgt = member.model._index_features(features[:, f_idx + 1], tgt, mask_tgt)\n                raw = member.model.predict_edges(\n                    feat_src, feat_tgt, src * ds_arr_by_device[str(dev)],\n                    tgt * ds_arr_by_device[str(dev)], pos_src, pos_tgt, mask_src, mask_tgt,\n                )[0]\n                member_logits.append(raw)\n            fused_logits = weighted_average(member_logits, weights, primary)\n            fused_probs = (\n                torch.softmax(fused_logits, dim=0)\n                if cfg.edge_activation == "softmax"\n                else torch.sigmoid(fused_logits)\n            )\n            probs = fused_probs.cpu().numpy()\n            for i, j in np.argwhere(probs > cfg.threshold):\n                gi, gj = s_src + int(i), s_tgt + int(j)\n                dist = float(np.linalg.norm(\n                    coords_so_far[gi, 1:].astype(np.float32) - coords_so_far[gj, 1:].astype(np.float32)\n                ))\n                all_edges.append((gi, gj, float(probs[i, j]), dist))\n        del encoded, fused_det, imgs_cpu\n        if division_member is not None:\n            del division_features, division_detection, division_images\n\n    coords = np.concatenate(coord_lists) if coord_lists else np.empty((0, 4), np.int16)\n    coords = coords.astype(np.float32)\n    coords[:, 1:] *= ds_arr\n    native_coords = (\n        np.concatenate([division_coord_by_t[t] for t in range(T)]).astype(np.int16)\n        if division_member is not None and division_coord_by_t\n        else np.empty((0, 4), np.int16)\n    )\n    native_counts = np.asarray(\n        [len(division_coord_by_t.get(t, ())) for t in range(T)], np.int32\n    )\n    native_offsets = np.concatenate(\n        [[0], np.cumsum(native_counts, dtype=np.int64)]\n    )\n    dtypes = {\n        "source_id": np.int64,\n        "target_id": np.int64,\n        "probability": np.float32,\n        "alternative_parent_probability": np.float32,\n        "is_target_winner": np.uint8,\n        "distance_um": np.float32,\n        "source_target_rank": np.int16,\n    }\n    sidecar = {}\n    for key, chunks in division_evidence.items():\n        sidecar[key] = (\n            np.concatenate(chunks).astype(dtypes[key], copy=False)\n            if chunks else np.empty(0, dtypes[key])\n        )\n    sidecar["native_node_coords"] = native_coords\n    sidecar["native_frame_offsets"] = native_offsets\n    sidecar["spacing_um"] = original_spacing\n    return coords.astype(np.int16), all_edges, sidecar\n\n\ndef predict(\n    data_dir, fold, splits_file, members, cfg, method, video_slice=None,\n    division_member=None, division_sidecar_dir=None,\n    division_pool_um=3.0, division_radius_um=20.0,\n    division_topk=16, division_min_probability=0.01,\n):\n    folds = json.loads(splits_file.read_text())\n    names = folds[fold]["test"]\n    if video_slice is not None:\n        names = names[video_slice]\n    from dataspec import PREDICTIONS_PATH\n    output_dir = PREDICTIONS_PATH / USERNAME / method / f"split_{fold}"\n    if output_dir.exists():\n        for old in output_dir.glob("*.geff"):\n            shutil.rmtree(old) if old.is_dir() else old.unlink()\n        for old in list(output_dir.glob("*.ready")) + list(output_dir.glob("*.ready.tmp")):\n            old.unlink()\n    output_dir.mkdir(parents=True, exist_ok=True)\n    if division_sidecar_dir is not None:\n        division_sidecar_dir.mkdir(parents=True, exist_ok=True)\n    print(\n        f"True A+B ensemble: {len(members)} graph members, {len(names)} datasets; "\n        f"native_model_c_evidence={division_member is not None}",\n        flush=True,\n    )\n    for name in tqdm(names, desc="Predicting true ensemble"):\n        coords, edges, sidecar = predict_video_true_ensemble(\n            members, data_dir / name, cfg,\n            division_member=division_member,\n            division_pool_um=division_pool_um,\n            division_radius_um=division_radius_um,\n            division_topk=division_topk,\n            division_min_probability=division_min_probability,\n        )\n        graph = base.build_graph(coords, edges)\n        if cfg.use_ilp and graph.num_edges() > 0:\n            solver = td.solvers.ILPSolver(\n                edge_weight=cfg.ilp_edge_weight * td.EdgeAttr("edge_prob"),\n                appearance_weight=cfg.ilp_appearance_weight,\n                disappearance_weight=cfg.ilp_disappearance_weight,\n                division_weight=cfg.ilp_division_weight,\n            )\n            with base.suppress_output():\n                graph = solver.solve(graph)\n        graph_path = output_dir / f"{name}.geff"\n        save_graph(graph, graph_path)\n        if division_sidecar_dir is not None and division_member is not None:\n            sidecar_path = division_sidecar_dir / f"{name}.npz"\n            temporary = division_sidecar_dir / f"{name}.npz.tmp"\n            with temporary.open("wb") as handle:\n                np.savez_compressed(handle, **sidecar)\n            temporary.replace(sidecar_path)\n        ready_path = output_dir / f"{name}.ready"\n        ready_tmp = output_dir / f"{name}.ready.tmp"\n        ready_tmp.write_text(json.dumps({"dataset": name, "graph": str(graph_path)}))\n        ready_tmp.replace(ready_path)\n        print(f"  {name}: nodes={graph.num_nodes()} edges={graph.num_edges()} READY", flush=True)\n    print(f"Saved {len(names)} true-ensemble graphs to {output_dir}", flush=True)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--data-dir", required=True)\n    p.add_argument("--splits", required=True)\n    p.add_argument("--split", type=int, default=0)\n    p.add_argument("--method", default=DEFAULT_METHOD)\n    for letter in "abc":\n        p.add_argument(f"--weights-{letter}", default="")\n        p.add_argument(f"--weight-{letter}", type=float, default=1.0 if letter in "ab" else 0.0)\n        p.add_argument(f"--det-logit-bias-{letter}", type=float, default=0.0)\n        p.add_argument(f"--device-{letter}", default="cuda:0" if letter != "b" else "cuda:1")\n    p.add_argument("--division-weights", default="")\n    p.add_argument("--division-device", default="cuda:0")\n    p.add_argument("--division-sidecar-dir", default="")\n    p.add_argument("--division-pool-um", type=float, default=3.0)\n    p.add_argument("--division-radius-um", type=float, default=20.0)\n    p.add_argument("--division-topk", type=int, default=16)\n    p.add_argument("--division-min-probability", type=float, default=0.01)\n    p.add_argument("--det-threshold", type=float, default=0.99)\n    p.add_argument("--edge-threshold", type=float, default=0.5,\n                   help="Minimum fused edge probability admitted to the ILP.")\n    p.add_argument("--slice", default="")\n    p.add_argument("--use-ilp", action="store_true")\n    p.add_argument("--ilp-edge-weight", type=float, default=-1.0)\n    p.add_argument("--ilp-appearance-weight", type=float, default=0.1)\n    p.add_argument("--ilp-disappearance-weight", type=float, default=0.1)\n    p.add_argument("--ilp-division-weight", type=float, default=1.5)\n    args = p.parse_args()\n    specs = []\n    for letter in "abc":\n        path, weight = getattr(args, f"weights_{letter}"), getattr(args, f"weight_{letter}")\n        if path and weight > 0:\n            specs.append((\n                letter.upper(), Path(path), weight,\n                getattr(args, f"det_logit_bias_{letter}"),\n                getattr(args, f"device_{letter}"),\n            ))\n    members = load_members(specs)\n    division_member = None\n    if args.division_weights:\n        division_member = load_members([(\n            "C_DIVISION_NATIVE", Path(args.division_weights), 1.0, 0.0,\n            args.division_device,\n        )])[0]\n    cfg = PredictConfig(\n        det_threshold=args.det_threshold, threshold=args.edge_threshold,\n        use_ilp=args.use_ilp,\n        ilp_edge_weight=args.ilp_edge_weight,\n        ilp_appearance_weight=args.ilp_appearance_weight,\n        ilp_disappearance_weight=args.ilp_disappearance_weight,\n        ilp_division_weight=args.ilp_division_weight,\n    )\n    video_slice = slice(*[int(x) if x else None for x in args.slice.split(":")]) if args.slice else None\n    predict(\n        Path(args.data_dir), args.split, Path(args.splits), members, cfg,\n        args.method, video_slice,\n        division_member=division_member,\n        division_sidecar_dir=(\n            Path(args.division_sidecar_dir) if args.division_sidecar_dir else None\n        ),\n        division_pool_um=args.division_pool_um,\n        division_radius_um=args.division_radius_um,\n        division_topk=args.division_topk,\n        division_min_probability=args.division_min_probability,\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
(REPO_DIR / "scripts" / "predict_true_ensemble.py").write_text(TRUE_ENSEMBLE_SCRIPT)
(REPO_DIR / "kaggle_test_splits_true_ensemble.json").write_text(
    json.dumps([{"split": 0, "train": [], "test": test_stems}], indent=2)
)

predict_cmd = [
    sys.executable, "scripts/predict_true_ensemble.py",
    "--data-dir", str(TEST_DIR),
    "--splits", "kaggle_test_splits_true_ensemble.json",
    "--split", "0",
    "--method", METHOD,
    "--det-threshold", str(DET_THRESHOLD),
    "--edge-threshold", str(EDGE_THRESHOLD),
    "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    "--division-weights", str(MODEL_C_CHECKPOINT),
    "--division-device", "cuda:0",
    "--division-sidecar-dir", str(MODEL_C_EVIDENCE_DIR),
    "--division-pool-um", "3.0",
    "--division-radius-um", "20.0",
    "--division-topk", "16",
    "--division-min-probability", "0.01",
]
for letter, member in zip("abc", ENSEMBLE_MODELS):
    if member["enabled"] and member["weight"] > 0:
        predict_cmd += [f"--weights-{letter}", member["path"],
                        f"--weight-{letter}", str(member["weight"]),
                        f"--device-{letter}", member["device"]]
    else:
        predict_cmd += [f"--weight-{letter}", "0"]
if USE_ILP:
    predict_cmd.append("--use-ilp")
if SLICE:
    predict_cmd += ["--slice", SLICE]

print("True ensemble members:")
for member in enabled_models:
    print(f"  {member['name']} weight={member['weight']} device={member['device']}")
print("A+B only -> fused detections -> continuation edges -> one ILP")
print("Native Model C -> division evidence only; decoder threshold=0.96")
PREDICTION_LOG_PATH = WORKING_DIR / "true_ensemble_prediction.log"
PREDICTION_LOG_HANDLE = PREDICTION_LOG_PATH.open("w", buffering=1)
PREDICT_START_WALL = time.time()
PREDICT_START_MONOTONIC = time.monotonic()
PREDICT_PROCESS = subprocess.Popen(
    predict_cmd, cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": "src"},
    stdout=PREDICTION_LOG_HANDLE, stderr=subprocess.STDOUT, text=True,
)
predict_seconds = float("nan")
print(
    f"Started true-ensemble producer pid={PREDICT_PROCESS.pid}; "
    f"streaming log={PREDICTION_LOG_PATH}", flush=True,
)


In [ ]:
# Clean official-event V2 selector: frozen V1 pair ranker -> leave-four-out parent/time gate -> tube NMS.
# Attach the `biohub-division-parent-gate-v2` Kaggle dataset before running.
DIVISION_GBM_ENABLED = os.environ.get("BIOHUB_DIVISION_GBM_ENABLED", "1") != "0"
DIVISION_GBM_THRESHOLD = float(os.environ.get("BIOHUB_DIVISION_GBM_THRESHOLD", "0.50"))
DIVISION_GBM_RESCUE_DELTA = float(os.environ.get("BIOHUB_DIVISION_GBM_RESCUE_DELTA", "0.15"))
DIVISION_GBM_STEAL_DELTA = float(os.environ.get("BIOHUB_DIVISION_GBM_STEAL_DELTA", "0.25"))
ANYTIME_HARD_LIMIT_SECONDS = float(os.environ.get("BIOHUB_ANYTIME_HARD_LIMIT_SECONDS", str(11 * 3600 + 50 * 60)))
ANYTIME_FINALIZE_RESERVE_SECONDS = float(os.environ.get("BIOHUB_ANYTIME_FINALIZE_RESERVE_SECONDS", "600"))
ANYTIME_MIN_UPGRADE_SECONDS = float(os.environ.get("BIOHUB_ANYTIME_MIN_UPGRADE_SECONDS", "60"))
DIVISION_GBM_ARTIFACT_DIR = Path(os.environ.get(
    "BIOHUB_DIVISION_GBM_ARTIFACT_DIR",
    "/kaggle/input/datasets/tweakai/biohub-model-c-combined-primary-v1",
))


def _find_division_gbm_artifact() -> Path:
    candidates = [
        DIVISION_GBM_ARTIFACT_DIR,
        Path("/kaggle/input/biohub-model-c-combined-primary-v1"),
        Path("/kaggle/input/datasets/tweakai/biohub-model-c-combined-primary-v1"),
    ]
    for candidate in candidates:
        if (candidate / "deploy_spec.json").exists() and (candidate / "division_gbm_runtime.py").exists():
            return candidate
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for spec_path in input_root.rglob("deploy_spec.json"):
            candidate = spec_path.parent
            if (candidate / "division_gbm_runtime.py").exists() and (candidate / "pair_model.joblib").exists():
                return candidate
    raise FileNotFoundError(
        "Attach the biohub-division-parent-gate-v2 dataset; deploy_spec.json and division_gbm_runtime.py were not found"
    )


DIVISION_GBM_RUNTIME = None
if DIVISION_GBM_ENABLED:
    DIVISION_GBM_ARTIFACT_DIR = _find_division_gbm_artifact()
    _division_spec = importlib.util.spec_from_file_location(
        "biohub_division_gbm_runtime",
        DIVISION_GBM_ARTIFACT_DIR / "division_gbm_runtime.py",
    )
    if _division_spec is None or _division_spec.loader is None:
        raise RuntimeError("Unable to load division_gbm_runtime.py")
    _division_module = importlib.util.module_from_spec(_division_spec)
    sys.modules[_division_spec.name] = _division_module
    _division_spec.loader.exec_module(_division_module)
    DIVISION_GBM_RUNTIME = _division_module.DivisionGBMRuntime(DIVISION_GBM_ARTIFACT_DIR)
    _safe_spec = importlib.util.spec_from_file_location(
        "biohub_model_c_safe_runtime",
        DIVISION_GBM_ARTIFACT_DIR / "model_c_safe_runtime.py",
    )
    if _safe_spec is None or _safe_spec.loader is None:
        raise RuntimeError("Unable to load model_c_safe_runtime.py")
    _safe_module = importlib.util.module_from_spec(_safe_spec)
    sys.modules[_safe_spec.name] = _safe_module
    _safe_spec.loader.exec_module(_safe_module)
    MODEL_C_PRIMARY_RUNTIME = _safe_module.CombinedModelCPrimaryRuntime(
        DIVISION_GBM_RUNTIME,
        DIVISION_GBM_ARTIFACT_DIR,
        threshold=MODEL_C_PRIMARY_THRESHOLD,
    )
    EXPERIMENT_TAG = "ab_v2_native_model_c_combined_primary_hidden"

print(json.dumps({
    "division_gbm_enabled": DIVISION_GBM_ENABLED,
    "division_gbm_artifact_dir": str(DIVISION_GBM_ARTIFACT_DIR),
    "division_gbm_threshold": DIVISION_GBM_THRESHOLD,
    "division_gbm_rescue_delta": DIVISION_GBM_RESCUE_DELTA,
    "division_gbm_steal_delta": DIVISION_GBM_STEAL_DELTA,
    "legacy_safe_divisions_replaced": bool(DIVISION_GBM_ENABLED),
    "model_c_native_evidence": True,
    "model_c_primary_threshold": MODEL_C_PRIMARY_THRESHOLD,
    "model_c_fused_graph_weight": 0.0,
    "anytime_baseline_first": True,
    "anytime_hard_limit_hours": ANYTIME_HARD_LIMIT_SECONDS / 3600.0,
    "anytime_finalize_reserve_minutes": ANYTIME_FINALIZE_RESERVE_SECONDS / 60.0,
    "anytime_min_upgrade_minutes": ANYTIME_MIN_UPGRADE_SECONDS / 60.0,
}, indent=2))


In [ ]:
import tracksdata as td
import numpy as np
import blosc2
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

SUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
CSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]
VOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)
DENSE_SHORT7_COMPONENT_ROWS: list[dict[str, object]] = []
DENSE_SHORT7_COMPONENT_MEMBERS: list[dict[str, object]] = []
DENSE_SHORT7_NODE_ROWS: list[dict[str, object]] = []
REGISTRATION_DIAGNOSTIC_ROWS: list[dict[str, object]] = []


def graph_from_geff(path: Path):
    graph = td.graph.IndexedRXGraph.from_geff(path)
    return graph[0] if isinstance(graph, tuple) else graph


def edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:
    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]
    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]
    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:
    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]
    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]
    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def node_point(node: dict[str, object]) -> tuple[float, float, float]:
    return (float(node["z"]), float(node["y"]), float(node["x"]))


def edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:
    prob = edge.get("edge_prob")
    prob_value = float(prob) if prob is not None else 0.0
    ranking_distance = edge.get("registered_distance_um", edge["distance_um"])
    return prob_value, -float(ranking_distance)


def _next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:
    return max(nodes_by_id) + 1 if nodes_by_id else 1



def read_test_frame(dataset: str, t: int, frame_cache: dict[int, np.ndarray]) -> np.ndarray:
    if t in frame_cache:
        return frame_cache[t]
    zarr_path = TEST_DIR / f"{dataset}.zarr"
    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())
    shape = tuple(int(v) for v in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    frame_shape = shape[1:]
    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    try:
        raw = chunk_path.read_bytes()
        arr = np.frombuffer(blosc2.decompress(raw), dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            frame = arr.reshape(frame_shape).copy()
            frame_cache[t] = frame
            return frame
    except Exception:
        pass
    import zarr
    frame = np.asarray(zarr.open(zarr_path / "0", mode="r")[t])
    frame_cache[t] = frame
    return frame


def _registration_view(frame: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return raw and normalized/downsampled registration volumes."""
    z_step = max(1, REGISTRATION_DOWNSAMPLE_Z)
    yx_step = max(1, REGISTRATION_DOWNSAMPLE_YX)
    raw = np.asarray(frame[::z_step, ::yx_step, ::yx_step])
    work = np.log1p(raw.astype(np.float32))
    work -= float(work.mean())
    std = float(work.std())
    if std > 1e-6:
        work /= std
    wz = np.hanning(work.shape[0]).astype(np.float32) if work.shape[0] > 2 else np.ones(work.shape[0], np.float32)
    wy = np.hanning(work.shape[1]).astype(np.float32) if work.shape[1] > 2 else np.ones(work.shape[1], np.float32)
    wx = np.hanning(work.shape[2]).astype(np.float32) if work.shape[2] > 2 else np.ones(work.shape[2], np.float32)
    work *= wz[:, None, None] * wy[None, :, None] * wx[None, None, :]
    return raw, work


def _phase_registration_displacement(reference: np.ndarray, moving: np.ndarray) -> tuple[np.ndarray, float]:
    """Estimate content displacement reference->moving and a peak-to-sidelobe score."""
    axes = tuple(range(reference.ndim))
    f_ref = np.fft.rfftn(reference, axes=axes)
    f_mov = np.fft.rfftn(moving, axes=axes)
    cross = f_ref * np.conj(f_mov)
    cross /= np.maximum(np.abs(cross), 1e-8)
    corr = np.fft.irfftn(cross, s=reference.shape, axes=axes).real
    peak_idx = np.array(np.unravel_index(int(np.argmax(corr)), corr.shape), dtype=np.int64)
    peak = float(corr[tuple(peak_idx)])
    sidelobe = corr.copy()
    for dz in range(-2, 3):
        for dy in range(-2, 3):
            for dx in range(-2, 3):
                idx = tuple(int((peak_idx[k] + d) % corr.shape[k]) for k, d in enumerate((dz, dy, dx)))
                sidelobe[idx] = np.nan
    side_mean = float(np.nanmean(sidelobe))
    side_std = float(np.nanstd(sidelobe))
    psr = (peak - side_mean) / max(side_std, 1e-8)
    alignment_shift = peak_idx.astype(np.float64)
    for axis, size in enumerate(corr.shape):
        if alignment_shift[axis] > size // 2:
            alignment_shift[axis] -= size
    # Phase correlation gives the shift applied to moving to align it to reference.
    # Negating it gives the physical content displacement reference -> moving.
    scale = np.array([max(1, REGISTRATION_DOWNSAMPLE_Z), max(1, REGISTRATION_DOWNSAMPLE_YX), max(1, REGISTRATION_DOWNSAMPLE_YX)], dtype=np.float64)
    return -alignment_shift * scale, float(psr)


def _registration_geometry_evidence(
    nodes_by_id: dict[int, dict[str, object]],
    t0: int,
    shift_um: np.ndarray,
) -> dict[str, float] | None:
    """Compare registered and unregistered nearest-neighbor residuals symmetrically."""
    if not REGISTRATION_GEOMETRY_VALIDATE:
        return None
    source = np.asarray([_position_um(node) for node in nodes_by_id.values() if int(node["t"]) == t0], dtype=np.float64)
    target = np.asarray([_position_um(node) for node in nodes_by_id.values() if int(node["t"]) == t0 + 1], dtype=np.float64)
    if len(source) < REGISTRATION_GEOMETRY_MIN_NODES or len(target) < REGISTRATION_GEOMETRY_MIN_NODES:
        return None
    source_tree = cKDTree(source)
    target_tree = cKDTree(target)
    raw_forward = target_tree.query(source, k=1)[0]
    raw_reverse = source_tree.query(target, k=1)[0]
    shifted_forward = target_tree.query(source + shift_um[None, :], k=1)[0]
    shifted_reverse = source_tree.query(target - shift_um[None, :], k=1)[0]
    raw = np.concatenate([raw_forward, raw_reverse])
    shifted = np.concatenate([shifted_forward, shifted_reverse])
    cap = max(0.1, REGISTRATION_GEOMETRY_DISTANCE_CAP_UM)
    raw_clipped = np.minimum(raw, cap)
    shifted_clipped = np.minimum(shifted, cap)
    # Median alone is insensitive in very dense frames; blend it with a capped mean.
    raw_score = 0.5 * float(np.median(raw_clipped)) + 0.5 * float(np.mean(raw_clipped))
    shifted_score = 0.5 * float(np.median(shifted_clipped)) + 0.5 * float(np.mean(shifted_clipped))
    coverage_um = max(0.1, REGISTRATION_GEOMETRY_COVERAGE_UM)
    raw_coverage = float(np.mean(raw <= coverage_um))
    shifted_coverage = float(np.mean(shifted <= coverage_um))
    return {
        "raw_score_um": raw_score,
        "shifted_score_um": shifted_score,
        "improvement_um": raw_score - shifted_score,
        "raw_coverage": raw_coverage,
        "shifted_coverage": shifted_coverage,
        "coverage_gain": shifted_coverage - raw_coverage,
        "source_nodes": float(len(source)),
        "target_nodes": float(len(target)),
    }


def estimate_frame_registration(
    dataset: str | None,
    times: list[int],
    stats: dict[str, object],
    nodes_by_id: dict[int, dict[str, object]],
) -> dict[int, np.ndarray]:
    """Return confidence-gated global displacement in microns for each t->t+1."""
    if not OUTPUT_REGISTRATION_AWARE or dataset is None or len(times) < 2:
        return {}
    frame_cache: dict[int, np.ndarray] = {}
    shifts_um: dict[int, np.ndarray] = {}
    candidate_magnitudes: list[float] = []
    applied_magnitudes: list[float] = []
    scale_um = np.asarray(VOXEL_SCALE_UM, dtype=np.float64)
    pairs = [(t, t + 1) for t in sorted(set(times)) if t + 1 in set(times)]
    previous_t = None
    previous_raw = None
    previous_view = None
    for t0, t1 in pairs:
        try:
            if previous_t != t0 or previous_raw is None or previous_view is None:
                raw0, view0 = _registration_view(read_test_frame(dataset, t0, frame_cache))
            else:
                raw0, view0 = previous_raw, previous_view
            raw1, view1 = _registration_view(read_test_frame(dataset, t1, frame_cache))
            duplicate = bool(np.array_equal(raw0, raw1))
            shift_vox, psr = _phase_registration_displacement(view0, view1)
            shift_um = shift_vox * scale_um
            magnitude_um = float(np.linalg.norm(shift_um))
            candidate_magnitudes.append(magnitude_um)
            confident = bool(np.isfinite(psr) and psr >= REGISTRATION_MIN_PSR)
            within_max = bool(magnitude_um <= REGISTRATION_MAX_APPLY_UM)
            abrupt = bool(magnitude_um >= REGISTRATION_MIN_APPLY_UM)
            base_applied = bool(confident and within_max and abrupt and not duplicate)
            geometry = _registration_geometry_evidence(nodes_by_id, t0, shift_um)
            geometry_vetoed = False
            geometry_rescued = False
            applied = base_applied
            if geometry is not None:
                stats["registration_geometry_validated_pairs"] += 1
                improvement = float(geometry["improvement_um"])
                coverage_gain = float(geometry["coverage_gain"])
                if base_applied:
                    geometry_vetoed = bool(
                        improvement <= -REGISTRATION_GEOMETRY_VETO_MARGIN_UM
                        and coverage_gain <= -REGISTRATION_GEOMETRY_VETO_COVERAGE_DROP
                    )
                    if geometry_vetoed:
                        applied = False
                        stats["registration_geometry_vetoed_pairs"] += 1
                else:
                    rescue_eligible = bool(
                        confident and within_max and not duplicate
                        and REGISTRATION_GEOMETRY_RESCUE_MIN_UM <= magnitude_um < REGISTRATION_MIN_APPLY_UM
                    )
                    geometry_rescued = bool(
                        rescue_eligible and (
                            (improvement >= REGISTRATION_GEOMETRY_RESCUE_MARGIN_UM and coverage_gain >= 0.0)
                            or (improvement >= 0.10 and coverage_gain >= REGISTRATION_GEOMETRY_RESCUE_COVERAGE_GAIN)
                        )
                    )
                    if geometry_rescued:
                        applied = True
                        stats["registration_geometry_rescued_pairs"] += 1
            if applied:
                shifts_um[t0] = shift_um
                applied_magnitudes.append(magnitude_um)
                stats["registration_applied_pairs"] += 1
            if duplicate:
                stats["registration_duplicate_pairs"] += 1
                reason = "duplicate"
            elif not confident:
                stats["registration_rejected_low_confidence"] += 1
                reason = "low_confidence"
            elif not within_max:
                stats["registration_rejected_large_shift"] += 1
                reason = "over_max"
            elif geometry_vetoed:
                reason = "geometry_veto"
            elif geometry_rescued:
                reason = "geometry_rescue"
            elif not abrupt:
                reason = "below_activation"
            else:
                reason = "applied"
            if confident:
                stats["registration_confident_pairs"] += 1
            stats["registration_pairs"] += 1
            REGISTRATION_DIAGNOSTIC_ROWS.append({
                "dataset": dataset, "t_from": t0, "t_to": t1,
                "shift_z_vox": float(shift_vox[0]), "shift_y_vox": float(shift_vox[1]), "shift_x_vox": float(shift_vox[2]),
                "shift_z_um": float(shift_um[0]), "shift_y_um": float(shift_um[1]), "shift_x_um": float(shift_um[2]),
                "shift_magnitude_um": magnitude_um, "psr": float(psr),
                "duplicate_frame": int(duplicate), "applied": int(applied), "reason": reason,
                "geometry_raw_score_um": float(geometry["raw_score_um"]) if geometry else np.nan,
                "geometry_shifted_score_um": float(geometry["shifted_score_um"]) if geometry else np.nan,
                "geometry_improvement_um": float(geometry["improvement_um"]) if geometry else np.nan,
                "geometry_raw_coverage": float(geometry["raw_coverage"]) if geometry else np.nan,
                "geometry_shifted_coverage": float(geometry["shifted_coverage"]) if geometry else np.nan,
                "geometry_coverage_gain": float(geometry["coverage_gain"]) if geometry else np.nan,
            })
            previous_t, previous_raw, previous_view = t1, raw1, view1
            frame_cache.clear()
        except Exception as exc:
            stats["registration_failed_pairs"] += 1
            REGISTRATION_DIAGNOSTIC_ROWS.append({
                "dataset": dataset, "t_from": t0, "t_to": t1, "applied": 0,
                "reason": f"error:{type(exc).__name__}",
            })
            previous_t = previous_raw = previous_view = None
            frame_cache.clear()
    stats["registration_candidate_shift_mean_um"] = float(np.mean(candidate_magnitudes)) if candidate_magnitudes else 0.0
    stats["registration_candidate_shift_max_um"] = float(np.max(candidate_magnitudes)) if candidate_magnitudes else 0.0
    stats["registration_applied_shift_mean_um"] = float(np.mean(applied_magnitudes)) if applied_magnitudes else 0.0
    stats["registration_applied_shift_max_um"] = float(np.max(applied_magnitudes)) if applied_magnitudes else 0.0
    return shifts_um


def refine_synthetic_midpoint(
    dataset: str | None,
    t: int,
    midpoint: tuple[float, float, float],
    frame_cache: dict[int, np.ndarray],
    stats: dict[str, int],
) -> tuple[float, float, float]:
    if not GAP_REFINE_SYNTHETIC or dataset is None:
        return midpoint
    try:
        frame = read_test_frame(dataset, t, frame_cache)
        z, y, x = [int(round(v)) for v in midpoint]
        z0 = max(0, z - GAP_REFINE_WIN_Z)
        z1 = min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)
        y0 = max(0, y - GAP_REFINE_WIN_YX)
        y1 = min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)
        x0 = max(0, x - GAP_REFINE_WIN_YX)
        x1 = min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)
        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]
        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]
        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]
        baseline = float(np.percentile(patch, GAP_REFINE_BG_PERCENTILE))
        weights = np.maximum(patch - baseline, 0.0)
        if float(weights.sum()) <= 0:
            # Preserve the original low-percentile fallback when the local peak is weak.
            baseline = float(np.percentile(patch, 20.0))
            weights = np.maximum(patch - baseline, 0.0)
        sigma = max(GAP_REFINE_CENTER_SIGMA_UM, 1e-6)
        dz_um = (zz - float(midpoint[0])) * VOXEL_SCALE_UM[0]
        dy_um = (yy - float(midpoint[1])) * VOXEL_SCALE_UM[1]
        dx_um = (xx - float(midpoint[2])) * VOXEL_SCALE_UM[2]
        center_prior = np.exp(-0.5 * (dz_um * dz_um + dy_um * dy_um + dx_um * dx_um) / (sigma * sigma))
        weights *= center_prior
        total = float(weights.sum())
        if total <= 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        refined = (
            float((weights * zz).sum() / total),
            float((weights * yy).sum() / total),
            float((weights * xx).sum() / total),
        )
        stats["gap_refine_centered"] += 1
        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:
            stats["gap_refine_rejected_shift"] += 1
            return midpoint
        stats["gap_refined_synthetic"] += 1
        return refined
    except Exception:
        stats["gap_refine_failed"] += 1
        return midpoint


def _position_um(node: dict[str, object]) -> np.ndarray:
    return np.array(
        [float(node["z"]) * VOXEL_SCALE_UM[0], float(node["y"]) * VOXEL_SCALE_UM[1], float(node["x"]) * VOXEL_SCALE_UM[2]],
        dtype=np.float64,
    )



import torch
from scipy.spatial import cKDTree

class _MotionResidual(torch.nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(n_features, 64), torch.nn.SiLU(), torch.nn.Dropout(.05),
            torch.nn.Linear(64, 32), torch.nn.SiLU(), torch.nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def _load_motion_corrector():
    if not MOTION_COST_CORRECTOR_ENABLED:
        return None
    if not MOTION_COST_CORRECTOR_PATH.exists():
        raise FileNotFoundError(f"Motion corrector not found: {MOTION_COST_CORRECTOR_PATH}")
    ck = torch.load(MOTION_COST_CORRECTOR_PATH, map_location="cpu", weights_only=False)
    expected = [
        "base_cost","raw_dist","registered_dist","motion_dist","abs_dz","abs_dy","abs_dx",
        "abs_reg_dz","abs_reg_dy","abs_reg_dx","velocity_z","velocity_y","velocity_x","velocity_mag",
        "shift_z","shift_y","shift_x","shift_mag","density_src","density_tgt",
        "z_boundary_src","z_boundary_tgt","has_predecessor",
    ]
    if list(ck["features"]) != expected:
        raise ValueError(f"Unexpected motion-corrector features: {ck['features']}")
    model = _MotionResidual(len(expected))
    model.load_state_dict(ck["model"])
    model.eval()
    print(f"Loaded learned motion cost: {MOTION_COST_CORRECTOR_PATH} J={ck['metrics']['jaccard']:.6f}")
    return {
        "model": model,
        "mean": torch.as_tensor(ck["mean"], dtype=torch.float32),
        "std": torch.as_tensor(ck["std"], dtype=torch.float32),
        "residual_scale": float(ck["residual_scale"]),
    }

MOTION_COST_CORRECTOR = _load_motion_corrector()

@torch.no_grad()
def _motion_cost_residual(features: list[list[float]]) -> np.ndarray:
    if MOTION_COST_CORRECTOR is None or not features:
        return np.zeros(len(features), dtype=np.float64)
    x = torch.as_tensor(np.asarray(features, dtype=np.float32))
    x = (x - MOTION_COST_CORRECTOR["mean"]) / MOTION_COST_CORRECTOR["std"]
    raw = MOTION_COST_CORRECTOR["model"](x)
    scale = MOTION_COST_CORRECTOR["residual_scale"]
    return (scale * torch.tanh(raw / scale)).cpu().numpy().astype(np.float64)


def motion_relink_edges(
    nodes_by_id: dict[int, dict[str, object]],
    stats: dict[str, int],
    learned_edge_probs: dict[tuple[int, int], float] | None = None,
    registration_shifts_um: dict[int, np.ndarray] | None = None,
) -> list[dict[str, object]]:
    if not OUTPUT_MOTION_RELINK or not nodes_by_id:
        return []

    learned_edge_probs = learned_edge_probs or {}
    registration_shifts_um = registration_shifts_um or {}

    def learned_prob(source_id: int, target_id: int) -> float:
        value = learned_edge_probs.get((source_id, target_id), 0.0)
        try:
            value = float(value)
        except (TypeError, ValueError):
            return 0.0
        if not np.isfinite(value):
            return 0.0
        if value < 0.0 or value > 1.0:
            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))
        return float(np.clip(value, 0.0, 1.0))

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)
    for ids in ids_by_t.values():
        ids.sort()

    frame_sizes = [len(ids) for ids in ids_by_t.values()]
    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:
        stats["motion_relink_skipped_large_frame"] = 1
        return []

    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}
    density_15um: dict[int, float] = {}
    for frame_ids in ids_by_t.values():
        if not frame_ids:
            continue
        frame_pos = np.stack([position_um[node_id] for node_id in frame_ids])
        tree = cKDTree(frame_pos)
        for node_id, count in zip(frame_ids, tree.query_ball_point(frame_pos, 15.0, return_length=True)):
            density_15um[node_id] = float(max(0, int(count) - 1))
    predecessor_position_um: dict[int, np.ndarray] = {}
    selected_edges: list[dict[str, object]] = []

    def assign_pass(
        source_ids: list[int],
        target_ids: list[int],
        gate_um: float,
    ) -> list[tuple[int, int, float, float, float, float]]:
        if not source_ids or not target_ids:
            return []
        big = gate_um * 1000.0 + 1.0
        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)
        raw_dist = np.full_like(cost, np.inf)
        motion_dist = np.full_like(cost, np.inf)
        prob_matrix = np.zeros_like(cost)
        registered_dist = np.full_like(cost, np.inf)
        source_t = int(nodes_by_id[source_ids[0]]["t"])
        global_shift = np.asarray(registration_shifts_um.get(source_t, np.zeros(3)), dtype=np.float64)
        correction_features: list[list[float]] = []
        correction_pairs: list[tuple[int, int]] = []
        previous_global_shift = np.asarray(registration_shifts_um.get(source_t - 1, np.zeros(3)), dtype=np.float64)
        for i, source_id in enumerate(source_ids):
            source_pos = position_um[source_id]
            prev_pos = predecessor_position_um.get(source_id)
            if prev_pos is None:
                local_velocity = np.zeros(3, dtype=np.float64)
                has_predecessor = 0.0
                predicted = source_pos + global_shift
            else:
                local_velocity = source_pos - prev_pos - previous_global_shift
                has_predecessor = 1.0
                predicted = source_pos + global_shift + MOTION_RELINK_VELOCITY_WEIGHT * local_velocity
            for j, target_id in enumerate(target_ids):
                target_pos = position_um[target_id]
                raw = float(np.linalg.norm(target_pos - source_pos))
                registered = float(np.linalg.norm(target_pos - source_pos - global_shift))
                if registered > gate_um:
                    continue
                motion = float(np.linalg.norm(target_pos - predicted))
                prob = learned_prob(source_id, target_id)
                raw_dist[i, j] = raw
                registered_dist[i, j] = registered
                motion_dist[i, j] = motion
                prob_matrix[i, j] = prob
                base_geometry_cost = motion + 0.05 * registered
                cost[i, j] = base_geometry_cost - MOTION_RELINK_LEARNED_BONUS * prob
                raw_delta_vec = target_pos - source_pos
                registered_delta_vec = raw_delta_vec - global_shift
                zmax = 102.375
                zsrc = float(np.clip(source_pos[0] / zmax, 0.0, 1.0))
                ztgt = float(np.clip(target_pos[0] / zmax, 0.0, 1.0))
                correction_features.append([
                    base_geometry_cost, raw, registered, motion,
                    abs(raw_delta_vec[0]), abs(raw_delta_vec[1]), abs(raw_delta_vec[2]),
                    abs(registered_delta_vec[0]), abs(registered_delta_vec[1]), abs(registered_delta_vec[2]),
                    local_velocity[0], local_velocity[1], local_velocity[2], float(np.linalg.norm(local_velocity)),
                    global_shift[0], global_shift[1], global_shift[2], float(np.linalg.norm(global_shift)),
                    density_15um.get(source_id, 0.0), density_15um.get(target_id, 0.0),
                    min(zsrc, 1.0-zsrc), min(ztgt, 1.0-ztgt), has_predecessor,
                ])
                correction_pairs.append((i, j))
        if MOTION_COST_CORRECTOR is not None and correction_features:
            residuals = _motion_cost_residual(correction_features) * MOTION_COST_CORRECTOR_STRENGTH
            for (i, j), residual in zip(correction_pairs, residuals):
                cost[i, j] -= float(residual)
            stats["motion_corrector_candidates"] += len(correction_pairs)
            stats["motion_corrector_abs_residual_sum"] += float(np.abs(residuals).sum())
        row_ind, col_ind = linear_sum_assignment(cost)
        matches: list[tuple[int, int, float, float, float, float]] = []
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] >= big:
                continue
            matches.append((
                source_ids[int(r)],
                target_ids[int(c)],
                float(raw_dist[r, c]),
                float(registered_dist[r, c]),
                float(motion_dist[r, c]),
                float(prob_matrix[r, c]),
            ))
        return matches

    times = sorted(ids_by_t)
    for t in times:
        source_ids = ids_by_t.get(t, [])
        target_ids = ids_by_t.get(t + 1, [])
        if not source_ids or not target_ids:
            continue
        unmatched_sources = set(source_ids)
        unmatched_targets = set(target_ids)
        frame_matches: list[tuple[int, int, float, float, float, str, float]] = []
        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), ("relaxed", MOTION_RELINK_RELAXED_UM)):
            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]
            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]
            matches = assign_pass(pass_sources, pass_targets, gate_um)
            for source_id, target_id, raw, registered, motion, prob in matches:
                if source_id not in unmatched_sources or target_id not in unmatched_targets:
                    continue
                unmatched_sources.remove(source_id)
                unmatched_targets.remove(target_id)
                frame_matches.append((source_id, target_id, raw, registered, motion, pass_name, prob))
                if pass_name == "tight":
                    stats["motion_relink_tight_edges"] += 1
                else:
                    stats["motion_relink_relaxed_edges"] += 1
        for source_id, target_id, raw, registered, motion, pass_name, prob in frame_matches:
            if raw > (MOTION_RELINK_TIGHT_UM if pass_name == "tight" else MOTION_RELINK_RELAXED_UM):
                stats["registration_rescued_over_gate_edges"] += 1
            if int(nodes_by_id[source_id]["t"]) in registration_shifts_um:
                stats["registration_compensated_motion_edges"] += 1
            selected_edges.append({
                "source_id": source_id,
                "target_id": target_id,
                "edge_prob": prob,
                "distance_um": raw,
                "registered_distance_um": registered,
                "motion_distance_um": motion,
                "motion_relinked": 1,
                "motion_pass": pass_name,
            })
            predecessor_position_um[target_id] = position_um[source_id]
        stats["motion_relink_frames"] += 1

    stats["motion_relink_edges"] = len(selected_edges)
    return selected_edges

def close_single_frame_gaps(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
    registration_shifts_um: dict[int, np.ndarray] | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:
        return nodes_by_id, edges

    registration_shifts_um = registration_shifts_um or {}
    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    incident = outgoing | incoming

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    isolated_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)
        if node_id not in incident:
            isolated_by_t.setdefault(t, []).append(node_id)

    max_synthetic = min(
        GAP_CLOSE_MAX_ADDED_ABS,
        max(1, int(round(len(nodes_by_id) * GAP_CLOSE_MAX_ADDED_FRAC))) if GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0,
    )
    next_id = _next_node_id(nodes_by_id)
    frame_cache: dict[int, np.ndarray] = {}
    used_starts: set[int] = set()
    used_isolated: set[int] = set()
    synthetic_added = 0
    new_edges: list[dict[str, object]] = []

    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1)
    stats["gap_close_effective_max_gap"] = effective_gap_max
    for gap in range(1, effective_gap_max + 1):
        for t, end_ids in sorted(ends_by_t.items()):
            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]
            if not end_ids or not start_ids:
                continue

            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]
            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]
            threshold_um = GAP_CLOSE_UM * (gap + 1)
            d = np.zeros((len(end_ids), len(start_ids)), dtype=np.float64)
            for i, ep in enumerate(end_points):
                for j, sp in enumerate(start_points):
                    source_t = int(nodes_by_id[end_ids[i]]["t"])
                    total_global = sum((np.asarray(registration_shifts_um.get(source_t + step, np.zeros(3)), dtype=np.float64) for step in range(gap + 1)), np.zeros(3, dtype=np.float64))
                    ep_um = np.asarray(ep, dtype=np.float64) * np.asarray(VOXEL_SCALE_UM, dtype=np.float64)
                    sp_um = np.asarray(sp, dtype=np.float64) * np.asarray(VOXEL_SCALE_UM, dtype=np.float64)
                    d[i, j] = float(np.linalg.norm(sp_um - ep_um - total_global))
            stats["gap_candidates"] += int((d <= threshold_um).sum())
            if not np.isfinite(d).any():
                continue

            big = threshold_um * 1000.0 + 1.0
            cost = np.where(d <= threshold_um, d, big)
            row_ind, col_ind = linear_sum_assignment(cost)
            for r, c in zip(row_ind, col_ind):
                if d[r, c] > threshold_um:
                    continue
                source_id = end_ids[int(r)]
                target_id = start_ids[int(c)]
                if source_id in outgoing or target_id in used_starts:
                    continue

                source = nodes_by_id[source_id]
                target = nodes_by_id[target_id]
                mid_t = int(source["t"]) + gap
                source_um = _position_um(source)
                target_um = _position_um(target)
                total_global = sum((np.asarray(registration_shifts_um.get(int(source["t"]) + step, np.zeros(3)), dtype=np.float64) for step in range(gap + 1)), np.zeros(3, dtype=np.float64))
                first_global = np.asarray(registration_shifts_um.get(int(source["t"]), np.zeros(3)), dtype=np.float64)
                local_total = target_um - source_um - total_global
                mid_um = source_um + first_global + local_total / float(gap + 1)
                mid_vox = mid_um / np.asarray(VOXEL_SCALE_UM, dtype=np.float64)
                mid_point = tuple(float(v) for v in mid_vox)

                middle_id: int | None = None
                if GAP_CLOSE_REUSE_EXISTING:
                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]
                    if candidates:
                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) for nid in candidates]
                        best_idx = int(np.argmin(distances))
                        if distances[best_idx] <= GAP_CLOSE_REUSE_UM:
                            middle_id = candidates[best_idx]
                            used_isolated.add(middle_id)
                            stats["gap_reused_existing"] += 1

                if middle_id is None:
                    if synthetic_added >= max_synthetic:
                        stats["gap_skipped_node_cap"] += 1
                        continue
                    middle_id = next_id
                    next_id += 1
                    refined_point = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)
                    nodes_by_id[middle_id] = {
                        "node_id": middle_id,
                        "t": mid_t,
                        "z": refined_point[0],
                        "y": refined_point[1],
                        "x": refined_point[2],
                    }
                    synthetic_added += 1
                    stats["gap_inserted_synthetic"] += 1

                middle = nodes_by_id[middle_id]
                e1 = {
                    "source_id": source_id,
                    "target_id": middle_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(source, middle),
                    "gap_closed": 1,
                }
                e2 = {
                    "source_id": middle_id,
                    "target_id": target_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(middle, target),
                    "gap_closed": 1,
                }
                new_edges.extend([e1, e2])
                outgoing.add(source_id)
                incoming.add(middle_id)
                outgoing.add(middle_id)
                incoming.add(target_id)
                used_starts.add(target_id)
                stats["gap_pairs_selected"] += 1
                stats["gap_added_edges"] += 2

    if new_edges:
        edges = [*edges, *new_edges]
    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]
    return nodes_by_id, edges


def _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:
    by_source: dict[int, list[int]] = {}
    for edge in edges:
        by_source.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))
    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}


def _single_predecessor_map(edges: list[dict[str, object]]) -> dict[int, int]:
    by_target: dict[int, list[int]] = {}
    for edge in edges:
        by_target.setdefault(int(edge["target_id"]), []).append(int(edge["source_id"]))
    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}


def recover_strict_gap2(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:
        return nodes_by_id, edges

    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    predecessor = _single_predecessor_map(edges)
    successor = _single_successor_map(edges)

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)

    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * GAP2_MAX_LINKS_FRAC))))
    proposals: list[tuple[float, int, int, int, float]] = []

    def pos_um(node_id: int) -> np.ndarray:
        node = nodes_by_id[node_id]
        return np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64) * np.array(VOXEL_SCALE_UM)

    for t, end_ids in sorted(ends_by_t.items()):
        start_ids = starts_by_t.get(t + 3, [])
        if not end_ids or not start_ids:
            continue
        for end_id in end_ids:
            end_pos = pos_um(end_id)
            for start_id in start_ids:
                start_pos = pos_um(start_id)
                dist = float(np.linalg.norm(start_pos - end_pos))
                if dist > GAP2_MAX_TOTAL_UM or dist / 3.0 > GAP2_MAX_STEP_UM:
                    continue
                step = (start_pos - end_pos) / 3.0
                context_penalty = 0.0
                if GAP2_REQUIRE_CONTEXT:
                    ok_context = False
                    prev_id = predecessor.get(end_id)
                    if prev_id is not None:
                        prev_step = end_pos - pos_um(prev_id)
                        prev_norm = float(np.linalg.norm(prev_step))
                        step_norm = float(np.linalg.norm(step))
                        if prev_norm <= 0.01 or step_norm <= 0.01:
                            ok_context = True
                        else:
                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:
                                ok_context = True
                            context_penalty += max(0.0, 0.25 - cos)
                    next_id = successor.get(start_id)
                    if next_id is not None:
                        next_step = pos_um(next_id) - start_pos
                        next_norm = float(np.linalg.norm(next_step))
                        step_norm = float(np.linalg.norm(step))
                        if next_norm <= 0.01 or step_norm <= 0.01:
                            ok_context = True
                        else:
                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:
                                ok_context = True
                            context_penalty += max(0.0, 0.25 - cos)
                    if not ok_context:
                        continue
                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))

    proposals.sort(key=lambda item: item[0])
    stats["gap2_candidates"] = len(proposals)
    if not proposals:
        return nodes_by_id, edges

    selected: list[tuple[float, int, int, int, float]] = []
    used_ends: set[int] = set()
    used_starts: set[int] = set()
    per_frame_count: dict[int, int] = {}
    for proposal in proposals:
        if len(selected) >= cap:
            stats["gap2_skipped_cap"] += 1
            break
        _, end_id, start_id, t, _ = proposal
        if end_id in used_ends or start_id in used_starts:
            continue
        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * GAP2_FRAME_FRAC_CAP)))
        if per_frame_count.get(t, 0) >= frame_cap:
            continue
        selected.append(proposal)
        used_ends.add(end_id)
        used_starts.add(start_id)
        per_frame_count[t] = per_frame_count.get(t, 0) + 1

    if not selected:
        return nodes_by_id, edges

    next_node_id = _next_node_id(nodes_by_id)
    frame_cache: dict[int, np.ndarray] = {}
    new_edges: list[dict[str, object]] = []
    for _, end_id, start_id, t, _ in selected:
        source = nodes_by_id[end_id]
        target = nodes_by_id[start_id]
        previous_id = end_id
        inserted_ids: list[int] = []
        for k in (1, 2):
            frac = k / 3.0
            mid_t = int(source["t"]) + k
            midpoint = (
                float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,
                float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,
                float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,
            )
            refined_point = refine_synthetic_midpoint(dataset, mid_t, midpoint, frame_cache, stats)
            node_id = next_node_id
            next_node_id += 1
            nodes_by_id[node_id] = {
                "node_id": node_id,
                "t": mid_t,
                "z": refined_point[0],
                "y": refined_point[1],
                "x": refined_point[2],
            }
            inserted_ids.append(node_id)
            current = nodes_by_id[node_id]
            new_edges.append({
                "source_id": previous_id,
                "target_id": node_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(nodes_by_id[previous_id], current),
                "gap2_recovered": 1,
            })
            previous_id = node_id
        new_edges.append({
            "source_id": previous_id,
            "target_id": start_id,
            "edge_prob": None,
            "distance_um": edge_distance_um(nodes_by_id[previous_id], target),
            "gap2_recovered": 1,
        })
        stats["gap2_pairs_selected"] += 1
        stats["gap2_added_nodes"] += len(inserted_ids)
        stats["gap2_added_edges"] += 3

    return nodes_by_id, [*edges, *new_edges]


def add_safe_divisions_postlink(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    registration_shifts_um: dict[int, np.ndarray] | None = None,
) -> list[dict[str, object]]:
    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:
        return edges

    registration_shifts_um = registration_shifts_um or {}

    def registered_parent_distance(source: dict[str, object], child: dict[str, object]) -> float:
        t = int(source["t"])
        shift = np.asarray(registration_shifts_um.get(t, np.zeros(3)), dtype=np.float64)
        return float(np.linalg.norm(_position_um(child) - _position_um(source) - shift))

    out_by_source: dict[int, list[dict[str, object]]] = {}
    incoming: set[int] = set()
    for edge in edges:
        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)
        incoming.add(int(edge["target_id"]))

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)

    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}
    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))
    added: list[dict[str, object]] = []
    used_targets: set[int] = set()

    for t in sorted(ids_by_t):
        child_frame_ids = ids_by_t.get(t + 1, [])
        if not child_frame_ids:
            continue
        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]
        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]
        if not source_ids or not candidate_ids:
            continue

        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))
        proposals: list[tuple[float, int, int, float, float]] = []
        for source_id in source_ids:
            source = nodes_by_id[source_id]
            existing_child_edge = out_by_source[source_id][0]
            existing_child_id = int(existing_child_edge["target_id"])
            existing_child = nodes_by_id.get(existing_child_id)
            if existing_child is None or int(existing_child["t"]) != t + 1:
                continue
            child_dist = registered_parent_distance(source, existing_child)
            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:
                continue
            for candidate_id in candidate_ids:
                if (source_id, candidate_id) in existing_edges:
                    continue
                candidate = nodes_by_id[candidate_id]
                parent_dist = registered_parent_distance(source, candidate)
                if parent_dist > SAFE_DIV_MAX_UM:
                    continue
                sister_dist = edge_distance_um(existing_child, candidate)
                if sister_dist > SAFE_DIV_SISTER_MAX_UM:
                    continue
                score = parent_dist + 0.15 * sister_dist
                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))

        stats["safe_division_candidates"] += len(proposals)
        if not proposals:
            continue
        proposals.sort(key=lambda item: item[0])
        added_this_frame = 0
        for _, source_id, candidate_id, parent_dist, _ in proposals:
            if len(added) >= global_cap:
                stats["safe_division_skipped_cap"] += 1
                break
            if added_this_frame >= frame_cap:
                break
            if candidate_id in used_targets or candidate_id in incoming:
                continue
            candidate = nodes_by_id[candidate_id]
            added.append({
                "source_id": source_id,
                "target_id": candidate_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(nodes_by_id[source_id], candidate),
                "registered_distance_um": parent_dist,
                "safe_division": 1,
            })
            used_targets.add(candidate_id)
            added_this_frame += 1

    if added:
        stats["safe_divisions_added"] = len(added)
        return [*edges, *added]
    return edges


def filter_short_track_components(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN <= 1 or not edges:
        return nodes_by_id, edges

    times = [int(node["t"]) for node in nodes_by_id.values()]
    frame_span = max(1, max(times) - min(times) + 1) if times else 1
    mean_nodes_per_frame = len(nodes_by_id) / frame_span
    effective_min_track_len = OUTPUT_MIN_TRACK_LEN
    if OUTPUT_ADAPTIVE_MIN_TRACK_LEN and mean_nodes_per_frame < OUTPUT_DENSE_NODES_PER_FRAME:
        effective_min_track_len = max(OUTPUT_MIN_TRACK_LEN, OUTPUT_LOW_DENSITY_MIN_TRACK_LEN)
    stats["short_track_effective_min_len"] = int(effective_min_track_len)
    stats["short_track_mean_nodes_per_frame"] = float(mean_nodes_per_frame)
    stats["short_track_adaptive_low_density"] = int(effective_min_track_len != OUTPUT_MIN_TRACK_LEN)

    parent = {node_id: node_id for node_id in nodes_by_id}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(a: int, b: int) -> None:
        if a not in parent or b not in parent:
            return
        ra = find(a)
        rb = find(b)
        if ra != rb:
            parent[ra] = rb

    out_count: dict[int, int] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        union(source_id, target_id)
        out_count[source_id] = out_count.get(source_id, 0) + 1

    components: dict[int, list[int]] = {}
    for node_id in nodes_by_id:
        components.setdefault(find(node_id), []).append(node_id)

    root_by_node = {node_id: find(node_id) for node_id in nodes_by_id}
    component_edges: dict[int, list[dict[str, object]]] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source_root = root_by_node.get(source_id)
        if source_root is not None and source_root == root_by_node.get(target_id):
            component_edges.setdefault(source_root, []).append(edge)

    if RUN_DENSE_SHORT7_DIAGNOSTICS and dataset is not None and mean_nodes_per_frame >= OUTPUT_DENSE_NODES_PER_FRAME:
        sequence_t_min = min(times) if times else 0
        sequence_t_max = max(times) if times else 0
        dense_component_count = 0
        dense_removable_count = 0
        for root, members in components.items():
            if len(members) != 7:
                continue
            member_ids = sorted(members, key=lambda nid: (int(nodes_by_id[nid]["t"]), nid))
            member_set = set(member_ids)
            comp_edges = component_edges.get(root, [])
            has_division = any(out_count.get(node_id, 0) >= 2 for node_id in member_ids)
            protected_division = bool(OUTPUT_KEEP_DIVISION_COMPONENTS and has_division)
            would_remove_len8 = not protected_division

            edge_probs: list[float] = []
            raw_distances: list[float] = []
            motion_distances: list[float] = []
            for edge in comp_edges:
                for key, target in (("edge_prob", edge_probs), ("distance_um", raw_distances), ("motion_distance_um", motion_distances)):
                    value = edge.get(key)
                    if value is None:
                        continue
                    try:
                        value = float(value)
                    except (TypeError, ValueError):
                        continue
                    if np.isfinite(value):
                        target.append(value)

            ordered_nodes = [nodes_by_id[node_id] for node_id in member_ids]
            positions_um = np.stack([_position_um(node) for node in ordered_nodes])
            component_times = np.asarray([int(node["t"]) for node in ordered_nodes], dtype=np.float64)
            step_um = np.linalg.norm(np.diff(positions_um, axis=0), axis=1) if len(positions_um) >= 2 else np.asarray([])
            accel_um = np.linalg.norm(np.diff(positions_um, n=2, axis=0), axis=1) if len(positions_um) >= 3 else np.asarray([])
            if len(np.unique(component_times)) >= 3:
                fitted = np.stack([np.polyval(np.polyfit(component_times, positions_um[:, axis], 1), component_times) for axis in range(3)], axis=1)
                residual_um = np.linalg.norm(positions_um - fitted, axis=1)
            else:
                residual_um = np.asarray([])

            z_values = [float(node["z"]) for node in ordered_nodes]
            y_values = [float(node["y"]) for node in ordered_nodes]
            x_values = [float(node["x"]) for node in ordered_nodes]
            component_id = f"{dataset}:{min(member_ids)}"
            edge_pairs = ";".join(f"{int(edge['source_id'])}:{int(edge['target_id'])}" for edge in comp_edges)
            DENSE_SHORT7_COMPONENT_ROWS.append({
                "dataset": dataset,
                "component_id": component_id,
                "component_size": len(member_ids),
                "edge_count": len(comp_edges),
                "node_ids": ";".join(str(node_id) for node_id in member_ids),
                "edge_pairs": edge_pairs,
                "t_min": int(component_times.min()),
                "t_max": int(component_times.max()),
                "t_span": int(component_times.max() - component_times.min() + 1),
                "touches_sequence_start": int(component_times.min() == sequence_t_min),
                "touches_sequence_end": int(component_times.max() == sequence_t_max),
                "near_z_boundary": int(min(z_values) <= 1.0 or max(z_values) >= 62.0),
                "near_yx_boundary": int(min(y_values) <= 3.0 or min(x_values) <= 3.0 or max(y_values) >= 252.0 or max(x_values) >= 252.0),
                "has_division": int(has_division),
                "protected_division": int(protected_division),
                "would_remove_len8": int(would_remove_len8),
                "learned_edge_count": len(edge_probs),
                "edge_prob_mean": float(np.mean(edge_probs)) if edge_probs else np.nan,
                "edge_prob_min": float(np.min(edge_probs)) if edge_probs else np.nan,
                "edge_prob_max": float(np.max(edge_probs)) if edge_probs else np.nan,
                "raw_distance_mean_um": float(np.mean(raw_distances)) if raw_distances else np.nan,
                "raw_distance_max_um": float(np.max(raw_distances)) if raw_distances else np.nan,
                "motion_distance_mean_um": float(np.mean(motion_distances)) if motion_distances else np.nan,
                "motion_distance_max_um": float(np.max(motion_distances)) if motion_distances else np.nan,
                "tight_edge_count": sum(edge.get("motion_pass") == "tight" for edge in comp_edges),
                "relaxed_edge_count": sum(edge.get("motion_pass") == "relaxed" for edge in comp_edges),
                "gap_edge_count": sum(bool(edge.get("gap_closed")) for edge in comp_edges),
                "gap2_edge_count": sum(bool(edge.get("gap2_recovered")) for edge in comp_edges),
                "safe_division_edge_count": sum(bool(edge.get("safe_division")) for edge in comp_edges),
                "step_mean_um": float(np.mean(step_um)) if step_um.size else np.nan,
                "step_max_um": float(np.max(step_um)) if step_um.size else np.nan,
                "accel_mean_um": float(np.mean(accel_um)) if accel_um.size else np.nan,
                "linefit_residual_mean_um": float(np.mean(residual_um)) if residual_um.size else np.nan,
                "linefit_residual_max_um": float(np.max(residual_um)) if residual_um.size else np.nan,
            })
            DENSE_SHORT7_COMPONENT_MEMBERS.append({"dataset": dataset, "component_id": component_id, "node_ids": member_ids})
            dense_component_count += 1
            dense_removable_count += int(would_remove_len8)
        stats["dense_short7_components"] = dense_component_count
        stats["dense_short7_would_remove_len8"] = dense_removable_count

    keep: set[int] = set()
    for members in components.values():
        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)
        if len(members) >= effective_min_track_len or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):
            keep.update(members)

    if not keep:
        stats["short_track_filter_skipped_all"] += 1
        return nodes_by_id, edges

    removed_nodes = len(nodes_by_id) - len(keep)
    if removed_nodes <= 0:
        return nodes_by_id, edges

    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}
    kept_edges = [
        edge for edge in edges
        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes
    ]
    stats["short_track_components_removed"] = sum(1 for members in components.values() if not (set(members) & keep))
    stats["short_track_nodes_removed"] = removed_nodes
    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges


def linefit_smooth_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    registration_shifts_um: dict[int, np.ndarray] | None = None,
) -> dict[int, dict[str, object]]:
    """Smooth linear track interiors without changing graph topology."""
    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT <= 0 or OUTPUT_LINEFIT_WINDOW <= 0 or not edges:
        return nodes_by_id

    predecessor: dict[int, list[int]] = {}
    successor: dict[int, list[int]] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None:
            continue
        if int(target["t"]) != int(source["t"]) + 1:
            continue
        successor.setdefault(source_id, []).append(target_id)
        predecessor.setdefault(target_id, []).append(source_id)

    original_pos = {
        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)
        for node_id, node in nodes_by_id.items()
    }
    registration_shifts_um = registration_shifts_um or {}
    cumulative_shift_vox: dict[int, np.ndarray] = {}
    times = sorted({int(node["t"]) for node in nodes_by_id.values()})
    if times:
        cumulative_shift_vox[times[0]] = np.zeros(3, dtype=np.float64)
        voxel_scale = np.asarray(VOXEL_SCALE_UM, dtype=np.float64)
        for t in range(times[0], times[-1]):
            cumulative_shift_vox[t + 1] = cumulative_shift_vox.get(t, np.zeros(3, dtype=np.float64)) + np.asarray(registration_shifts_um.get(t, np.zeros(3)), dtype=np.float64) / voxel_scale
    use_stabilized = bool(REGISTRATION_LINEFIT_STABILIZE and registration_shifts_um)
    fit_pos = {
        node_id: pos - cumulative_shift_vox.get(int(nodes_by_id[node_id]["t"]), np.zeros(3)) if use_stabilized else pos
        for node_id, pos in original_pos.items()
    }
    updated_pos: dict[int, np.ndarray] = {}
    weight = float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))

    for node_id in sorted(nodes_by_id):
        neighbourhood: list[tuple[int, int]] = [(0, node_id)]

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            prev_ids = predecessor.get(current, [])
            if len(prev_ids) != 1:
                break
            current = prev_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((-step, current))

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            next_ids = successor.get(current, [])
            if len(next_ids) != 1:
                break
            current = next_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((step, current))

        if len(neighbourhood) < 3:
            stats["linefit_skipped_nodes"] += 1
            continue

        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)
        coords = np.stack([fit_pos[nid] for _, nid in neighbourhood])
        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)
        if not np.isfinite(fitted).all():
            stats["linefit_skipped_nodes"] += 1
            continue
        if use_stabilized:
            fitted = fitted + cumulative_shift_vox.get(int(nodes_by_id[node_id]["t"]), np.zeros(3))
            stats["registration_linefit_stabilized_nodes"] += 1
        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted

    for node_id, pos in updated_pos.items():
        nodes_by_id[node_id]["z"] = float(pos[0])
        nodes_by_id[node_id]["y"] = float(pos[1])
        nodes_by_id[node_id]["x"] = float(pos[2])

    stats["linefit_smoothed_nodes"] = len(updated_pos)
    return nodes_by_id


def filter_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    raw_edges: list[dict[str, object]],
    dataset: str | None = None,
    division_mode: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:
    stats = {
        "raw_edges": len(raw_edges),
        "dropped_nonconsecutive_edges": 0,
        "dropped_long_edges": 0,
        "dropped_multi_parent_edges": 0,
        "dropped_multi_child_edges": 0,
        "dropped_division_edges": 0,
        "gap_candidates": 0,
        "gap_pairs_selected": 0,
        "gap_reused_existing": 0,
        "gap_inserted_synthetic": 0,
        "gap_added_nodes": 0,
        "gap_added_edges": 0,
        "gap_skipped_node_cap": 0,
        "gap_refined_synthetic": 0,
        "gap_refine_failed": 0,
        "gap_refine_rejected_shift": 0,
        "gap_refine_centered": 0,
        "pruned_isolated_nodes": 0,
        "motion_relink_edges": 0,
        "motion_relink_tight_edges": 0,
        "motion_relink_relaxed_edges": 0,
        "motion_relink_frames": 0,
        "motion_relink_replaced_raw_edges": 0,
        "motion_relink_fallback_raw": 0,
        "motion_relink_skipped_large_frame": 0,
        "motion_corrector_candidates": 0,
        "motion_corrector_abs_residual_sum": 0.0,
        "gap2_candidates": 0,
        "gap2_pairs_selected": 0,
        "gap2_added_nodes": 0,
        "gap2_added_edges": 0,
        "gap2_skipped_cap": 0,
        "safe_division_candidates": 0,
        "safe_divisions_added": 0,
        "safe_division_skipped_cap": 0,
        "short_track_components_removed": 0,
        "short_track_nodes_removed": 0,
        "short_track_edges_removed": 0,
        "short_track_filter_skipped_all": 0,
        "dense_short7_components": 0,
        "dense_short7_would_remove_len8": 0,
        "short_track_effective_min_len": 0,
        "short_track_mean_nodes_per_frame": 0.0,
        "short_track_adaptive_low_density": 0,
        "linefit_smoothed_nodes": 0,
        "linefit_skipped_nodes": 0,
        "registration_pairs": 0,
        "registration_confident_pairs": 0,
        "registration_applied_pairs": 0,
        "registration_duplicate_pairs": 0,
        "registration_rejected_low_confidence": 0,
        "registration_rejected_large_shift": 0,
        "registration_geometry_validated_pairs": 0,
        "registration_geometry_vetoed_pairs": 0,
        "registration_geometry_rescued_pairs": 0,
        "registration_failed_pairs": 0,
        "registration_candidate_shift_mean_um": 0.0,
        "registration_candidate_shift_max_um": 0.0,
        "registration_applied_shift_mean_um": 0.0,
        "registration_applied_shift_max_um": 0.0,
        "registration_compensated_motion_edges": 0,
        "registration_rescued_over_gate_edges": 0,
        "registration_linefit_stabilized_nodes": 0,
    }

    registration_times = sorted({int(node["t"]) for node in nodes_by_id.values()})
    registration_shifts_um = estimate_frame_registration(dataset, registration_times, stats, nodes_by_id)

    edges: list[dict[str, object]] = []
    for edge in raw_edges:
        source = nodes_by_id.get(int(edge["source_id"]))
        target = nodes_by_id.get(int(edge["target_id"]))
        if source is None or target is None:
            continue
        if OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:
            stats["dropped_nonconsecutive_edges"] += 1
            continue
        distance_um = edge_distance_um(source, target)
        edge["distance_um"] = distance_um
        source_t = int(source["t"])
        registration_shift = np.asarray(registration_shifts_um.get(source_t, np.zeros(3)), dtype=np.float64)
        registered_distance_um = float(np.linalg.norm(_position_um(target) - _position_um(source) - registration_shift))
        edge["registered_distance_um"] = registered_distance_um
        gate_distance_um = registered_distance_um if source_t in registration_shifts_um else distance_um
        if OUTPUT_EDGE_MAX_UM > 0 and gate_distance_um > OUTPUT_EDGE_MAX_UM:
            stats["dropped_long_edges"] += 1
            continue
        edges.append(edge)

    if OUTPUT_MOTION_RELINK:
        learned_edge_probs: dict[tuple[int, int], float] = {}
        for edge in edges:
            prob = edge.get("edge_prob")
            if prob is None:
                continue
            try:
                prob = float(prob)
            except (TypeError, ValueError):
                continue
            if np.isfinite(prob):
                key = (int(edge["source_id"]), int(edge["target_id"]))
                learned_edge_probs[key] = max(learned_edge_probs.get(key, float("-inf")), prob)
        motion_edges = motion_relink_edges(nodes_by_id, stats, learned_edge_probs, registration_shifts_um)
        if motion_edges:
            stats["motion_relink_replaced_raw_edges"] = len(edges)
            edges = motion_edges
        else:
            stats["motion_relink_fallback_raw"] = 1

    if OUTPUT_SINGLE_PARENT_REPAIR and edges:
        best_by_target: dict[int, dict[str, object]] = {}
        for edge in edges:
            target_id = int(edge["target_id"])
            prev = best_by_target.get(target_id)
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_target[target_id] = edge
        kept_ids = {id(edge) for edge in best_by_target.values()}
        stats["dropped_multi_parent_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids]

    if OUTPUT_SINGLE_CHILD_REPAIR and edges:
        best_by_source: dict[int, dict[str, object]] = {}
        for edge in edges:
            source_id = int(edge["source_id"])
            prev = best_by_source.get(source_id)
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_source[source_id] = edge
        kept_ids = {id(edge) for edge in best_by_source.values()}
        stats["dropped_multi_child_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids]

    nodes_by_id, edges = close_single_frame_gaps(nodes_by_id, edges, stats, dataset=dataset, registration_shifts_um=registration_shifts_um)
    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)
    use_division_gbm = DIVISION_GBM_ENABLED if division_mode is None else division_mode == "gbm"
    if use_division_gbm:
        if dataset is None:
            raise RuntimeError("Division GBM requires the dataset id")
        edges = MODEL_C_PRIMARY_RUNTIME.apply(
            TEST_DIR / f"{dataset}.zarr",
            MODEL_C_EVIDENCE_DIR / f"{dataset}.npz",
            nodes_by_id,
            edges,
            stats,
            registration_shifts_um=registration_shifts_um,
            spacing=VOXEL_SCALE_UM,
            v2_threshold=DIVISION_GBM_THRESHOLD,
            v2_rescue_delta=DIVISION_GBM_RESCUE_DELTA,
            v2_steal_delta=DIVISION_GBM_STEAL_DELTA,
        )
    else:
        edges = add_safe_divisions_postlink(nodes_by_id, edges, stats, registration_shifts_um)

    if OUTPUT_DIVISION_GEOMETRY_FILTER and edges:
        by_source: dict[int, list[dict[str, object]]] = {}
        for edge in edges:
            by_source.setdefault(int(edge["source_id"]), []).append(edge)

        filtered: list[dict[str, object]] = []
        for source_id, source_edges in by_source.items():
            if len(source_edges) <= 1:
                filtered.extend(source_edges)
                continue

            ranked = sorted(source_edges, key=edge_sort_key, reverse=True)
            source = nodes_by_id[source_id]
            top1 = ranked[0]
            top2 = ranked[1]
            d1 = float(top1.get("registered_distance_um", top1["distance_um"]))
            d2 = float(top2.get("registered_distance_um", top2["distance_um"]))
            sister = edge_distance_um(nodes_by_id[int(top1["target_id"])], nodes_by_id[int(top2["target_id"])])
            valid_division = (
                max(d1, d2) <= DIV_PARENT_MAX_UM
                and sister <= DIV_SISTER_MAX_UM
                and int(nodes_by_id[int(top1["target_id"])] ["t"]) == int(source["t"]) + 1
                and int(nodes_by_id[int(top2["target_id"])] ["t"]) == int(source["t"]) + 1
            )
            if valid_division:
                filtered.extend([top1, top2])
                stats["dropped_division_edges"] += max(0, len(ranked) - 2)
            elif DIV_DROP_TO_SINGLE_IF_BAD:
                filtered.append(top1)
                stats["dropped_division_edges"] += len(ranked) - 1
            else:
                filtered.extend(ranked)
        edges = filtered

    if OUTPUT_PRUNE_ISOLATED:
        incident = {int(edge["source_id"]) for edge in edges} | {int(edge["target_id"]) for edge in edges}
        if incident:
            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}
            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)
            nodes_by_id = kept_nodes
            edges = [edge for edge in edges if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id]

    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats, dataset=dataset)
    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats, registration_shifts_um)

    return nodes_by_id, edges, stats


def graph_nodes_edges(graph):
    nodes = {}
    for row in graph.node_attrs().iter_rows(named=True):
        nid = int(row["node_id"])
        nodes[nid] = {"node_id":nid, "t":int(row["t"]), "z":float(row["z"]),
                      "y":float(row["y"]), "x":float(row["x"])}
    edges = []
    for row in graph.edge_attrs().iter_rows(named=True):
        p = row.get("edge_prob") if hasattr(row, "get") else None
        edges.append({"source_id":int(row["source_id"]), "target_id":int(row["target_id"]),
                      "edge_prob":None if p is None else float(p)})
    return nodes, edges

def _discover_ready_graphs() -> dict[str, Path]:
    ready: dict[str, Path] = {}
    for marker in (REPO_DIR / "predictions").glob(f"*/{METHOD}/split_0/*.ready"):
        graph_path = marker.with_suffix(".geff")
        if graph_path.exists() and marker.stem in test_stems:
            ready[marker.stem] = graph_path
    return ready

print("Streaming scheduler waiting for atomic A+B ready markers", flush=True)

ANYTIME_SHARD_DIR = WORKING_DIR / "anytime_submission_shards"
if ANYTIME_SHARD_DIR.exists():
    shutil.rmtree(ANYTIME_SHARD_DIR)
ANYTIME_SHARD_DIR.mkdir(parents=True, exist_ok=True)


def _shard_path(dataset: str) -> Path:
    return ANYTIME_SHARD_DIR / f"{dataset}.csv"


def _write_dataset_shard(
    dataset: str,
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
) -> tuple[int, int, int]:
    destination = _shard_path(dataset)
    temporary = destination.with_suffix(".csv.tmp")
    local_id = 0
    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        for node_id in sorted(nodes_by_id):
            node = nodes_by_id[node_id]
            writer.writerow({
                "id": local_id, "dataset": dataset, "row_type": "node",
                "node_id": int(node["node_id"]), "t": int(node["t"]),
                "z": int(round(float(node["z"]))),
                "y": int(round(float(node["y"]))),
                "x": int(round(float(node["x"]))),
                "source_id": -1, "target_id": -1,
            })
            local_id += 1
        division_sources: dict[int, int] = {}
        for edge in edges:
            source_id, target_id = int(edge["source_id"]), int(edge["target_id"])
            if source_id not in nodes_by_id or target_id not in nodes_by_id:
                raise AssertionError(f"{dataset}: dangling edge after filtering")
            writer.writerow({
                "id": local_id, "dataset": dataset, "row_type": "edge",
                "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
                "source_id": source_id, "target_id": target_id,
            })
            local_id += 1
            division_sources[source_id] = division_sources.get(source_id, 0) + 1
    temporary.replace(destination)
    return len(nodes_by_id), len(edges), sum(count >= 2 for count in division_sources.values())


def _process_graph(geff_path: Path, division_mode: str):
    dataset = geff_path.stem
    graph = graph_from_geff(geff_path)
    nodes_by_id, raw_edges = graph_nodes_edges(graph)
    raw_node_count = len(nodes_by_id)
    started = time.monotonic()
    nodes_by_id, edges, filter_stats = filter_output_graph(
        nodes_by_id, raw_edges, dataset=dataset, division_mode=division_mode,
    )
    seconds = time.monotonic() - started
    if not nodes_by_id:
        raise AssertionError(f"{dataset}: post-processing removed every node")
    if RUN_DENSE_SHORT7_DIAGNOSTICS:
        for component in DENSE_SHORT7_COMPONENT_MEMBERS:
            if component["dataset"] != dataset:
                continue
            for component_node_id in component["node_ids"]:
                component_node = nodes_by_id.get(int(component_node_id))
                if component_node is None:
                    continue
                DENSE_SHORT7_NODE_ROWS.append({
                    "dataset": dataset,
                    "component_id": component["component_id"],
                    "node_id": int(component_node_id),
                    "t": int(component_node["t"]),
                    "z": float(component_node["z"]),
                    "y": float(component_node["y"]),
                    "x": float(component_node["x"]),
                })
    node_count, edge_count, division_count = _write_dataset_shard(dataset, nodes_by_id, edges)
    filter_stats.update({
        "true_ensemble_members": len(enabled_models),
        "true_ensemble_weight_sum": float(sum(model["weight"] for model in enabled_models)),
        "true_ensemble_single_ilp": 1,
        "anytime_division_mode": division_mode,
        "anytime_process_seconds": seconds,
    })
    row = {
        "dataset": dataset,
        "raw_nodes": raw_node_count,
        "nodes": node_count,
        "raw_edges": filter_stats["raw_edges"],
        "edges": edge_count,
        "division_like_sources": division_count,
        "edge_to_node_ratio": edge_count / max(node_count, 1),
        "gap_added_nodes_frac": filter_stats.get("gap_added_nodes", 0) / max(raw_node_count, 1),
        **filter_stats,
    }
    return row


# FOUR-WORKER STREAMING SCHEDULER. The GPU producer writes one atomic .ready
# marker per completed graph. The parent always creates the valid A+B baseline
# first. Independent CPU processes then run the exact V2 gate path and atomically
# replace only the video they finish successfully.
import multiprocessing as mp
import queue as stdlib_queue
import traceback
from threadpoolctl import threadpool_limits

stats_by_dataset: dict[str, dict[str, object]] = {}
geff_by_dataset: dict[str, Path] = {}
baseline_done: set[str] = set()
attempted_upgrades: set[str] = set()
upgrade_seconds: list[float] = []
upgraded: list[str] = []
skipped: list[str] = []

DIVISION_WORKERS = max(1, min(int(os.environ.get("BIOHUB_DIVISION_WORKERS", "4")), 4))
DIVISION_WORKERS_WHILE_PREDICTING = max(
    1,
    min(
        int(os.environ.get("BIOHUB_DIVISION_WORKERS_WHILE_PREDICTING", str(DIVISION_WORKERS))),
        DIVISION_WORKERS,
    ),
)
DIVISION_WORKER_THREADS = 1
STREAMING_WARMUP_GRAPHS = max(
    1, min(int(os.environ.get("BIOHUB_STREAMING_WARMUP_GRAPHS", "1")), len(test_stems))
)
processing_deadline = ANYTIME_HARD_LIMIT_SECONDS - ANYTIME_FINALIZE_RESERVE_SECONDS
processing_deadline_monotonic = NOTEBOOK_WALL_START + processing_deadline


class _AnytimeUpgradeDeadline(TimeoutError):
    pass


def _anytime_alarm_handler(signum, frame):
    raise _AnytimeUpgradeDeadline("Full-GBM upgrade reached the submission-safe deadline")


def _division_worker_main(worker_id: int, task_queue, result_queue) -> None:
    # Four Python processes are useful only if native libraries do not create
    # additional thread pools inside every process. Apply both environment and
    # runtime threadpool limits; the scoring code itself is otherwise untouched.
    for variable in (
        "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
        "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS",
    ):
        os.environ[variable] = str(DIVISION_WORKER_THREADS)

    signal.signal(signal.SIGALRM, _anytime_alarm_handler)
    with threadpool_limits(limits=DIVISION_WORKER_THREADS):
        while True:
            task = task_queue.get()
            if task is None:
                return
            dataset, graph_text = task
            started = time.monotonic()
            remaining = processing_deadline_monotonic - started
            if remaining <= 0:
                result_queue.put({
                    "worker_id": worker_id, "dataset": dataset,
                    "status": "deadline", "seconds": 0.0,
                    "error": "deadline reached before dispatch",
                })
                continue
            signal.setitimer(signal.ITIMER_REAL, max(remaining, 1.0))
            try:
                row = _process_graph(Path(graph_text), "gbm")
                result_queue.put({
                    "worker_id": worker_id, "dataset": dataset,
                    "status": "ok", "seconds": time.monotonic() - started,
                    "row": row,
                })
            except _AnytimeUpgradeDeadline as error:
                result_queue.put({
                    "worker_id": worker_id, "dataset": dataset,
                    "status": "deadline", "seconds": time.monotonic() - started,
                    "error": str(error),
                })
            except BaseException as error:
                result_queue.put({
                    "worker_id": worker_id, "dataset": dataset,
                    "status": "error", "seconds": time.monotonic() - started,
                    "error": f"{type(error).__name__}: {error}",
                    "traceback": traceback.format_exc(limit=12),
                })
            finally:
                signal.setitimer(signal.ITIMER_REAL, 0.0)
                # These diagnostics are intentionally retained from the parent
                # baseline pass, matching the prior sequential scheduler. Child
                # copies are discarded so persistent workers do not grow memory.
                REGISTRATION_DIAGNOSTIC_ROWS.clear()
                DENSE_SHORT7_COMPONENT_ROWS.clear()
                DENSE_SHORT7_COMPONENT_MEMBERS.clear()
                DENSE_SHORT7_NODE_ROWS.clear()


if os.name != "posix":
    raise RuntimeError("The four-worker notebook requires Kaggle's POSIX/fork runtime")

MP_CONTEXT = mp.get_context("fork")
DIVISION_RESULT_QUEUE = MP_CONTEXT.Queue()
DIVISION_TASK_QUEUES = [MP_CONTEXT.Queue(maxsize=1) for _ in range(DIVISION_WORKERS)]
DIVISION_PROCESSES = []
worker_tasks: dict[int, str | None] = {worker_id: None for worker_id in range(DIVISION_WORKERS)}
for worker_id in range(DIVISION_WORKERS):
    process = MP_CONTEXT.Process(
        target=_division_worker_main,
        args=(worker_id, DIVISION_TASK_QUEUES[worker_id], DIVISION_RESULT_QUEUE),
        name=f"division-v2-{worker_id}",
    )
    process.start()
    DIVISION_PROCESSES.append(process)

print(
    f"Started {DIVISION_WORKERS} isolated combined V2+Model C CPU workers "
    f"({DIVISION_WORKERS_WHILE_PREDICTING} active while A+B runs, "
    f"{DIVISION_WORKER_THREADS} native thread each)",
    flush=True,
)


def _collect_division_results() -> int:
    collected = 0
    while True:
        try:
            result = DIVISION_RESULT_QUEUE.get_nowait()
        except stdlib_queue.Empty:
            break
        collected += 1
        worker_id = int(result["worker_id"])
        dataset = str(result["dataset"])
        worker_tasks[worker_id] = None
        duration = float(result.get("seconds", 0.0))
        if result["status"] == "ok":
            row = result["row"]
            stats_by_dataset[dataset] = row
            upgraded.append(dataset)
            upgrade_seconds.append(duration)
            print(
                f"[parallel upgrade {len(upgraded):3d}] worker={worker_id} {dataset}: "
                f"{duration / 60:.1f}m wall="
                f"{(time.monotonic() - NOTEBOOK_WALL_START) / 3600:.2f}h",
                flush=True,
            )
        else:
            print(
                f"[parallel fallback] worker={worker_id} {dataset}: "
                f"{result['status']} {result.get('error', '')}; preserving A+B baseline",
                flush=True,
            )
            if result.get("traceback"):
                print(result["traceback"], flush=True)
    return collected


def _active_worker_count() -> int:
    return sum(dataset is not None for dataset in worker_tasks.values())


while True:
    # Baseline catch-up has priority. No video enters the optional queue before
    # its complete, atomic, score-safe A+B shard exists.
    ready = _discover_ready_graphs()
    geff_by_dataset.update(ready)
    new_baselines = sorted(set(ready) - baseline_done)
    for dataset in new_baselines:
        geff_path = ready[dataset]
        row = _process_graph(geff_path, "baseline")
        stats_by_dataset[dataset] = row
        baseline_done.add(dataset)
        print(
            f"[stream baseline {len(baseline_done):3d}/{len(test_stems)}] {dataset}: "
            f"nodes={row['nodes']:,} edges={row['edges']:,} "
            f"wall={(time.monotonic() - NOTEBOOK_WALL_START) / 3600:.2f}h",
            flush=True,
        )

    _collect_division_results()
    producer_returncode = PREDICT_PROCESS.poll()
    producer_done = producer_returncode is not None
    elapsed = time.monotonic() - NOTEBOOK_WALL_START
    remaining = processing_deadline - elapsed
    warm = len(baseline_done) >= STREAMING_WARMUP_GRAPHS or producer_done
    active_limit = DIVISION_WORKERS if producer_done else DIVISION_WORKERS_WHILE_PREDICTING

    # Detect an unexpected hard worker exit. Its current video safely remains
    # on the A+B shard; the notebook continues with surviving workers.
    for worker_id, process in enumerate(DIVISION_PROCESSES):
        if process.exitcode is not None and worker_tasks[worker_id] is not None:
            dataset = str(worker_tasks[worker_id])
            worker_tasks[worker_id] = None
            print(
                f"[parallel worker exit] worker={worker_id} dataset={dataset} "
                f"exit={process.exitcode}; preserving A+B baseline",
                flush=True,
            )

    upgrade_candidates = sorted(
        (
            geff_by_dataset[dataset]
            for dataset in baseline_done
            if dataset not in attempted_upgrades
        ),
        key=lambda path: (
            float(stats_by_dataset[path.stem].get(
                "short_track_mean_nodes_per_frame", float("inf")
            )),
            int(stats_by_dataset[path.stem].get("raw_nodes", 0)),
            path.stem,
        ),
    )

    if warm and remaining > ANYTIME_MIN_UPGRADE_SECONDS:
        idle_worker_ids = [
            worker_id
            for worker_id in range(active_limit)
            if worker_tasks[worker_id] is None
            and DIVISION_PROCESSES[worker_id].is_alive()
        ]
        for worker_id, geff_path in zip(idle_worker_ids, upgrade_candidates):
            dataset = geff_path.stem
            attempted_upgrades.add(dataset)
            worker_tasks[worker_id] = dataset
            DIVISION_TASK_QUEUES[worker_id].put((dataset, str(geff_path)))
            print(
                f"[parallel dispatch] worker={worker_id} {dataset} "
                f"active={_active_worker_count()}/{active_limit} "
                f"producer={'done' if producer_done else 'running'}",
                flush=True,
            )

    all_baselines = len(baseline_done) == len(test_stems)
    remaining_candidates = bool(set(baseline_done) - attempted_upgrades)
    active_upgrades = _active_worker_count()
    if producer_done and all_baselines and not remaining_candidates and active_upgrades == 0:
        break
    if producer_done and not all_baselines and not new_baselines and active_upgrades == 0:
        break
    if remaining <= 0 and active_upgrades == 0 and producer_done:
        break
    time.sleep(2.0)

_collect_division_results()
for worker_id, task_queue in enumerate(DIVISION_TASK_QUEUES):
    if DIVISION_PROCESSES[worker_id].is_alive():
        task_queue.put(None)
for process in DIVISION_PROCESSES:
    process.join(timeout=10.0)
    if process.is_alive():
        process.terminate()
        process.join(timeout=5.0)

producer_returncode = PREDICT_PROCESS.wait()
PREDICTION_LOG_HANDLE.close()
predict_seconds = time.monotonic() - PREDICT_START_MONOTONIC
if producer_returncode != 0:
    tail = PREDICTION_LOG_PATH.read_text(errors="replace").splitlines()[-40:]
    raise RuntimeError(
        f"A+B producer failed with exit code {producer_returncode}:\n" + "\n".join(tail)
    )

missing_baselines = sorted(set(test_stems) - baseline_done)
if missing_baselines:
    raise RuntimeError(
        f"A+B completed but {len(missing_baselines)} baseline shards are missing: "
        f"{missing_baselines[:10]}"
    )
skipped = sorted(set(test_stems) - set(upgraded))
geffs = [geff_by_dataset[dataset] for dataset in sorted(test_stems)]
print(
    f"[parallel stream complete] producer={predict_seconds / 3600:.2f}h "
    f"workers={DIVISION_WORKERS} baselines={len(baseline_done)} "
    f"upgraded={len(upgraded)} fallback={len(skipped)} "
    f"wall={(time.monotonic() - NOTEBOOK_WALL_START) / 3600:.2f}h",
    flush=True,
)

# Stream the atomic shards into the one required submission file and assign a
# globally contiguous id.  No whole-submission DataFrame is held in RAM.
row_id = total_nodes = total_edges = 0
seen_datasets: set[str] = set()
with SUBMISSION_PATH.open("w", newline="") as output_handle:
    writer = csv.DictWriter(output_handle, fieldnames=CSV_COLUMNS)
    writer.writeheader()
    for dataset in sorted(test_stems):
        shard = _shard_path(dataset)
        if not shard.exists():
            raise FileNotFoundError(f"Missing anytime baseline shard: {shard}")
        seen_datasets.add(dataset)
        with shard.open(newline="") as input_handle:
            for row in csv.DictReader(input_handle):
                row["id"] = row_id
                writer.writerow(row)
                row_id += 1
                if row["row_type"] == "node":
                    total_nodes += 1
                else:
                    total_edges += 1

expected_datasets = set(test_stems)
if seen_datasets != expected_datasets:
    raise AssertionError({
        "missing": sorted(expected_datasets - seen_datasets)[:10],
        "extra": sorted(seen_datasets - expected_datasets)[:10],
    })
assert row_id == total_nodes + total_edges
assert total_nodes > 0
header = SUBMISSION_PATH.open().readline().strip().split(",")
assert header == CSV_COLUMNS, f"Bad CSV header: {header}"

stats_rows = [stats_by_dataset[dataset] for dataset in sorted(stats_by_dataset)]
stats = pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)
stats["predict_minutes_total"] = predict_seconds / 60.0
stats["experiment_tag"] = EXPERIMENT_TAG
stats["anytime_wall_hours_final"] = (time.monotonic() - NOTEBOOK_WALL_START) / 3600.0
stats["anytime_upgraded_total"] = len(upgraded)
stats["anytime_division_workers"] = DIVISION_WORKERS
stats["anytime_division_workers_while_predicting"] = DIVISION_WORKERS_WHILE_PREDICTING
stats["anytime_baseline_total"] = len(test_stems) - len(upgraded)
stats.to_csv(RUN_STATS_PATH, index=False)
pd.DataFrame(REGISTRATION_DIAGNOSTIC_ROWS).to_csv(REGISTRATION_DIAGNOSTICS_PATH, index=False)
if RUN_DENSE_SHORT7_DIAGNOSTICS:
    pd.DataFrame(DENSE_SHORT7_COMPONENT_ROWS).to_csv(DENSE_SHORT7_COMPONENTS_PATH, index=False)
    pd.DataFrame(DENSE_SHORT7_NODE_ROWS).to_csv(DENSE_SHORT7_NODES_PATH, index=False)

print(f"Wrote {SUBMISSION_PATH} with {row_id:,} rows")
print(f"Node rows: {total_nodes:,} | edge rows: {total_edges:,}")
print(f"Anytime full-GBM upgrades: {len(upgraded):,}/{len(test_stems):,}")
print(f"Final wall time: {(time.monotonic() - NOTEBOOK_WALL_START) / 3600:.2f}h")
print(f"Wrote {RUN_STATS_PATH}")
display(stats.describe(include="all"))
display(pd.read_csv(SUBMISSION_PATH, nrows=8))


In [ ]:
if RUN_OUTPUT_DIAGNOSTICS and len(stats):
    import matplotlib.pyplot as plt

    stats_view = stats.copy()
    stats_view["dropped_edges_total"] = 0
    for col in [
        "dropped_nonconsecutive_edges",
        "dropped_long_edges",
        "dropped_multi_parent_edges",
        "dropped_multi_child_edges",
        "dropped_division_edges",
    ]:
        if col in stats_view:
            stats_view["dropped_edges_total"] += stats_view[col].fillna(0)
    stats_view["kept_edge_fraction"] = stats_view["edges"] / stats_view["raw_edges"].clip(lower=1)

    fig, axes = plt.subplots(2, 3, figsize=(14.0, 7.2), constrained_layout=True)
    axes = axes.ravel()

    axes[0].hist(stats_view["nodes"], bins=min(24, max(5, len(stats_view))))
    axes[0].set_title("Nodes per video")
    axes[0].set_xlabel("nodes")
    axes[0].set_ylabel("videos")

    axes[1].hist(stats_view["edges"], bins=min(24, max(5, len(stats_view))))
    axes[1].set_title("Edges per video")
    axes[1].set_xlabel("edges")

    axes[2].scatter(stats_view["nodes"], stats_view["edges"], s=30, alpha=0.75)
    axes[2].set_title("Edge count vs node count")
    axes[2].set_xlabel("nodes")
    axes[2].set_ylabel("edges")

    axes[3].hist(stats_view["edge_to_node_ratio"], bins=min(24, max(5, len(stats_view))))
    axes[3].set_title("Edges per node")
    axes[3].set_xlabel("edge_to_node_ratio")
    axes[3].set_ylabel("videos")

    axes[4].scatter(stats_view["raw_edges"], stats_view["edges"], s=30, alpha=0.75)
    axes[4].plot(
        [0, max(stats_view["raw_edges"].max(), 1)],
        [0, max(stats_view["raw_edges"].max(), 1)],
        color="black", linewidth=1, alpha=0.35,
    )
    axes[4].set_title("Raw vs kept edges")
    axes[4].set_xlabel("raw edges")
    axes[4].set_ylabel("kept edges")

    axes[5].hist(stats_view["division_like_sources"], bins=min(20, max(5, len(stats_view))))
    axes[5].set_title("Division-like sources")
    axes[5].set_xlabel("sources with out-degree >= 2")

    plt.show()

    drop_cols = [c for c in stats_view.columns if c.startswith("dropped_")]
    if drop_cols:
        drop_summary = stats_view[["dataset", "raw_edges", "edges", "kept_edge_fraction", *drop_cols]].copy()
        display(drop_summary.sort_values("dropped_edges_total", ascending=False).head(12))

    gap_cols = [c for c in stats_view.columns if c.startswith("gap_")]
    if gap_cols:
        gap_summary = stats_view[["dataset", "nodes", "edges", *gap_cols]].copy()
        display(gap_summary.sort_values("gap_added_edges", ascending=False).head(12))
        display(gap_summary[gap_cols].sum().to_frame("total").T)

    display(stats_view.sort_values("nodes", ascending=False).head(10))

    repair_cols = [
        "motion_relink_edges", "motion_relink_tight_edges", "motion_relink_relaxed_edges",
        "motion_relink_replaced_raw_edges", "motion_relink_fallback_raw",
        "gap_added_nodes", "gap_added_edges", "gap2_candidates",
        "gap2_pairs_selected", "gap2_added_nodes", "gap2_added_edges",
        "safe_division_candidates", "safe_divisions_added",
        "short_track_nodes_removed", "short_track_edges_removed",
    ]
    repair_cols = [col for col in repair_cols if col in stats_view.columns]
    if repair_cols:
        repair_summary = stats_view[repair_cols].sum().rename("total").to_frame()
        display(repair_summary)



In [ ]:
%%writefile /kaggle/working/biohub_local_gt_eval.py
from __future__ import annotations

import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment


COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((p for p in COMP_DIR_CANDIDATES if p.exists()), COMP_DIR_CANDIDATES[0])
TRAIN_DIR = Path(os.environ.get("BIOHUB_TRAIN_DIR", COMP_DIR / "train"))
SUBMISSION_PATH = Path(os.environ.get("BIOHUB_SUBMISSION_PATH", "/kaggle/working/submission.csv"))
OUT_PATH = Path(os.environ.get("BIOHUB_LOCAL_GT_EVAL_PATH", "/kaggle/working/local_gt_eval.csv"))
SHORT7_COMPONENTS_PATH = Path(os.environ.get("BIOHUB_SHORT7_COMPONENTS_PATH", "/kaggle/working/dense_short7_components.csv"))
SHORT7_NODES_PATH = Path(os.environ.get("BIOHUB_SHORT7_NODES_PATH", "/kaggle/working/dense_short7_nodes.csv"))

VOXEL_SCALE_UM = np.asarray((1.625, 0.40625, 0.40625), dtype=np.float32)
MATCH_UM = float(os.environ.get("BIOHUB_MATCH_UM", "7.0"))
NODE_PENALTY_A = float(os.environ.get("BIOHUB_NODE_PENALTY_A", "0.1"))


def graph_from_geff(path: Path):
    try:
        import tracksdata as td
    except Exception as e:
        raise RuntimeError(
            "tracksdata is required. Run this after the .901 notebook dependency install, "
            "or install the Biohub support-pack wheels first."
        ) from e
    graph = td.graph.IndexedRXGraph.from_geff(path)
    return graph[0] if isinstance(graph, tuple) else graph


def graph_nodes_edges(graph):
    nodes = {}
    for row in graph.node_attrs().iter_rows(named=True):
        nid = int(row["node_id"])
        nodes[nid] = {
            "node_id": nid,
            "t": int(row["t"]),
            "z": float(row["z"]),
            "y": float(row["y"]),
            "x": float(row["x"]),
        }
    edges = []
    for row in graph.edge_attrs().iter_rows(named=True):
        edges.append((int(row["source_id"]), int(row["target_id"])))
    return nodes, edges


def load_gt(dataset: str):
    geff_path = TRAIN_DIR / f"{dataset}.geff"
    if not geff_path.exists():
        raise FileNotFoundError(f"GT geff not found for {dataset}: {geff_path}")
    return graph_nodes_edges(graph_from_geff(geff_path))


def estimated_true_nodes_from_geff(dataset: str, fallback_gt_nodes: int) -> int:
    """Read the coarse total-node estimate used by the adjusted metric when present.

    The sparse GEFF contains annotated nodes only, but its metadata may include a
    coarse estimate of all true nodes. If unavailable, fall back to annotated GT
    count so the column remains defined; fallback is only a local-practice proxy.
    """
    geff_path = TRAIN_DIR / f"{dataset}.geff"
    # Optional manual override for quick Kaggle experiments:
    # BIOHUB_ESTIMATED_TRUE_NODES_JSON='{"44b6_0113de3b":33900,...}'
    override_json = os.environ.get("BIOHUB_ESTIMATED_TRUE_NODES_JSON", "").strip()
    if override_json:
        try:
            import json

            override = json.loads(override_json)
            value = override.get(dataset)
            if value is not None:
                value = int(round(float(value)))
                if value > 0:
                    return value
        except Exception:
            pass

    meta_path = geff_path / "zarr.json"
    if meta_path.exists():
        try:
            import json

            meta = json.loads(meta_path.read_text())
            # Known competition metadata path used by the Biohub notebooks.
            value = (
                meta.get("attributes", {})
                .get("geff", {})
                .get("extra", {})
                .get("estimated_number_of_nodes")
            )
            if value is not None:
                value = int(round(float(value)))
                if value > 0:
                    return value

            attrs = meta.get("attributes", {})
            # Accept a few likely names; GEFF metadata has varied across versions.
            stack = [attrs]
            if isinstance(attrs.get("geff"), dict):
                stack.append(attrs["geff"])
                if isinstance(attrs["geff"].get("extra"), dict):
                    stack.append(attrs["geff"]["extra"])
            if isinstance(attrs.get("metadata"), dict):
                stack.append(attrs["metadata"])
            keys = (
                "estimated_number_of_nodes",
                "estimated_nodes",
                "estimated_total_nodes",
                "total_nodes_estimate",
                "node_count_estimate",
                "true_node_count_estimate",
                "estimated_true_nodes",
            )
            for obj in stack:
                for key in keys:
                    value = obj.get(key)
                    if value is not None:
                        value = int(round(float(value)))
                        if value > 0:
                            return value

            # Last-resort recursive search for any key containing both
            # "estimated" and "node". This avoids silently using sparse count
            # when GEFF metadata shape changes.
            def walk(obj):
                if isinstance(obj, dict):
                    for k, v in obj.items():
                        lk = str(k).lower()
                        if "estimated" in lk and "node" in lk and v is not None:
                            try:
                                iv = int(round(float(v)))
                                if iv > 0:
                                    return iv
                            except Exception:
                                pass
                        found = walk(v)
                        if found is not None:
                            return found
                elif isinstance(obj, list):
                    for item in obj:
                        found = walk(item)
                        if found is not None:
                            return found
                return None

            found = walk(meta)
            if found is not None:
                return found
        except Exception:
            pass
    print(
        f"[warn] {dataset}: full-node estimate not found in {geff_path}/zarr.json; "
        f"falling back to sparse GT node count={fallback_gt_nodes}. "
        "Adjusted node penalty will be too harsh for this dataset.",
        flush=True,
    )
    return int(fallback_gt_nodes)


def load_pred(df: pd.DataFrame, dataset: str):
    g = df[df["dataset"] == dataset]
    node_rows = g[g["row_type"] == "node"]
    edge_rows = g[g["row_type"] == "edge"]
    nodes = {}
    for row in node_rows.itertuples(index=False):
        nid = int(row.node_id)
        nodes[nid] = {
            "node_id": nid,
            "t": int(row.t),
            "z": float(row.z),
            "y": float(row.y),
            "x": float(row.x),
        }
    edges = []
    for row in edge_rows.itertuples(index=False):
        s, t = int(row.source_id), int(row.target_id)
        if s in nodes and t in nodes:
            edges.append((s, t))
    return nodes, edges


def node_xyz_um(nodes: dict[int, dict], ids: list[int]) -> np.ndarray:
    arr = np.asarray([[nodes[i]["z"], nodes[i]["y"], nodes[i]["x"]] for i in ids], dtype=np.float32)
    return arr * VOXEL_SCALE_UM[None, :]


def match_nodes(pred_nodes: dict[int, dict], gt_nodes: dict[int, dict], gate_um: float = MATCH_UM):
    pred_by_t, gt_by_t = {}, {}
    for nid, node in pred_nodes.items():
        pred_by_t.setdefault(int(node["t"]), []).append(nid)
    for nid, node in gt_nodes.items():
        gt_by_t.setdefault(int(node["t"]), []).append(nid)

    pred_to_gt = {}
    gt_to_pred = {}
    dists = []
    for t in sorted(set(pred_by_t) | set(gt_by_t)):
        pids = pred_by_t.get(t, [])
        gids = gt_by_t.get(t, [])
        if not pids or not gids:
            continue
        P = node_xyz_um(pred_nodes, pids)
        G = node_xyz_um(gt_nodes, gids)
        C = np.linalg.norm(P[:, None, :] - G[None, :, :], axis=2)
        ri, ci = linear_sum_assignment(C)
        for r, c in zip(ri, ci):
            dist = float(C[r, c])
            if dist <= gate_um:
                pid, gid = pids[int(r)], gids[int(c)]
                pred_to_gt[pid] = gid
                gt_to_pred[gid] = pid
                dists.append(dist)
    return pred_to_gt, gt_to_pred, dists


def eval_dataset(dataset: str, pred_nodes: dict[int, dict], pred_edges: list[tuple[int, int]]):
    gt_nodes, gt_edges = load_gt(dataset)
    pred_to_gt, gt_to_pred, dists = match_nodes(pred_nodes, gt_nodes)

    gt_edge_set = set(gt_edges)
    pred_edge_set = set(pred_edges)
    gt_out = {}
    gt_in = {}
    for s, t in gt_edge_set:
        gt_out.setdefault(s, set()).add(t)
        gt_in.setdefault(t, set()).add(s)

    mapped_pred_edges = []
    edge_tp = 0
    edge_fp = 0
    ignored_pred_edges = 0

    for s, t in pred_edge_set:
        ms = pred_to_gt.get(s)
        mt = pred_to_gt.get(t)
        if ms is None and mt is None:
            ignored_pred_edges += 1
            continue
        mapped = (ms, mt)
        if ms is not None and mt is not None and mapped in gt_edge_set:
            edge_tp += 1
            mapped_pred_edges.append(mapped)
            continue
        # Metric FP rule: wrong edge is only penalized if it conflicts with an
        # annotated source or target neighborhood. Edges entirely outside the
        # annotated graph are ignored.
        source_conflict = ms is not None and ms in gt_out
        target_conflict = mt is not None and mt in gt_in
        if source_conflict or target_conflict:
            edge_fp += 1
            if ms is not None and mt is not None:
                mapped_pred_edges.append(mapped)
        else:
            ignored_pred_edges += 1

    edge_fn = max(0, len(gt_edge_set) - edge_tp)

    node_tp = len(pred_to_gt)
    pred_n, gt_n = len(pred_nodes), len(gt_nodes)
    pred_e, gt_e = len(pred_edge_set), len(gt_edge_set)
    t_true_est = estimated_true_nodes_from_geff(dataset, fallback_gt_nodes=gt_n)

    node_precision = node_tp / pred_n if pred_n else 0.0
    node_recall = node_tp / gt_n if gt_n else 0.0
    node_f1 = 2 * node_precision * node_recall / (node_precision + node_recall) if (node_precision + node_recall) else 0.0

    # The visible/practice GEFF annotations are sparse: most real cells are not marked.
    # Metric-style edge Jaccard penalizes only edges that conflict with annotated GT
    # neighborhoods, while ignoring edges outside annotated regions.
    strict_edge_jaccard_debug = edge_tp / (gt_e + pred_e - edge_tp) if (gt_e + pred_e - edge_tp) else 0.0
    metric_edge_jaccard_proxy = (
        edge_tp / (edge_tp + edge_fp + edge_fn)
        if (edge_tp + edge_fp + edge_fn)
        else 0.0
    )
    strict_edge_precision_debug = edge_tp / pred_e if pred_e else 0.0
    sparse_edge_recall = edge_tp / gt_e if gt_e else 0.0
    node_penalty_factor = max(0.0, 1.0 - NODE_PENALTY_A * max(0, pred_n - t_true_est) / max(1, t_true_est))
    adjusted_edge_jaccard_proxy = metric_edge_jaccard_proxy * node_penalty_factor

    gt_div_sources = {}
    pred_div_sources = {}
    for s, t in gt_edge_set:
        gt_div_sources[s] = gt_div_sources.get(s, 0) + 1
    for s, t in pred_edge_set:
        pred_div_sources[s] = pred_div_sources.get(s, 0) + 1
    gt_div_count = sum(1 for v in gt_div_sources.values() if v >= 2)
    pred_div_count = sum(1 for v in pred_div_sources.values() if v >= 2)

    return {
        "dataset": dataset,
        "pred_nodes": pred_n,
        "gt_nodes": gt_n,
        "node_tp": node_tp,
        "node_precision": node_precision,
        "node_recall": node_recall,
        "node_f1": node_f1,
        "mean_match_um": float(np.mean(dists)) if dists else math.nan,
        "p95_match_um": float(np.percentile(dists, 95)) if dists else math.nan,
        "pred_edges": pred_e,
        "gt_edges": gt_e,
        "edge_tp": edge_tp,
        "edge_fp_metric": edge_fp,
        "edge_fn_metric": edge_fn,
        "ignored_pred_edges_metric": ignored_pred_edges,
        "sparse_edge_recall": sparse_edge_recall,
        "metric_edge_jaccard_proxy": metric_edge_jaccard_proxy,
        "estimated_true_nodes": t_true_est,
        "node_penalty_factor_proxy": node_penalty_factor,
        "adjusted_edge_jaccard_proxy": adjusted_edge_jaccard_proxy,
        "strict_edge_precision_debug": strict_edge_precision_debug,
        "strict_edge_jaccard_debug": strict_edge_jaccard_debug,
        "pred_division_sources": pred_div_count,
        "gt_division_sources": gt_div_count,
    }


def annotate_dense_short7_components(submission_df: pd.DataFrame) -> None:
    if not SHORT7_COMPONENTS_PATH.exists() or not SHORT7_NODES_PATH.exists():
        print("[short7] component diagnostics not found; skipping GT annotation")
        return
    components = pd.read_csv(SHORT7_COMPONENTS_PATH)
    component_nodes = pd.read_csv(SHORT7_NODES_PATH)
    if components.empty or component_nodes.empty:
        print("[short7] no dense seven-node components found")
        return

    for column, default in (
        ("visible_gt_matched_nodes", 0),
        ("visible_gt_edge_tp", 0),
        ("visible_gt_edge_fp", 0),
        ("visible_gt_mean_match_um", np.nan),
        ("contains_visible_gt", 0),
    ):
        components[column] = default

    for dataset in sorted(set(components["dataset"].astype(str))):
        gt_path = TRAIN_DIR / f"{dataset}.geff"
        if not gt_path.exists():
            continue
        full_pred_nodes, _ = load_pred(submission_df, dataset)
        gt_nodes, gt_edges = load_gt(dataset)
        pred_to_gt, _, _ = match_nodes(full_pred_nodes, gt_nodes)
        gt_edge_set = set(gt_edges)
        gt_out: dict[int, set[int]] = {}
        gt_in: dict[int, set[int]] = {}
        for source_id, target_id in gt_edge_set:
            gt_out.setdefault(source_id, set()).add(target_id)
            gt_in.setdefault(target_id, set()).add(source_id)

        dataset_components = components.index[components["dataset"].astype(str) == dataset]
        for idx in dataset_components:
            component_id = str(components.at[idx, "component_id"])
            node_rows = component_nodes[component_nodes["component_id"].astype(str) == component_id]
            component_node_ids = {int(value) for value in node_rows["node_id"].tolist()}
            matched_node_ids = [node_id for node_id in component_node_ids if node_id in pred_to_gt]
            match_distances = []
            for pred_id in matched_node_ids:
                gt_id = pred_to_gt[pred_id]
                pred_pos = node_xyz_um(full_pred_nodes, [pred_id])[0]
                gt_pos = node_xyz_um(gt_nodes, [gt_id])[0]
                match_distances.append(float(np.linalg.norm(pred_pos - gt_pos)))

            edge_tp = 0
            edge_fp = 0
            edge_pairs_text = components.at[idx, "edge_pairs"]
            if pd.notna(edge_pairs_text):
                for pair in str(edge_pairs_text).split(";"):
                    if not pair or ":" not in pair:
                        continue
                    source_text, target_text = pair.split(":", 1)
                    source_id, target_id = int(source_text), int(target_text)
                    mapped_source = pred_to_gt.get(source_id)
                    mapped_target = pred_to_gt.get(target_id)
                    if mapped_source is not None and mapped_target is not None and (mapped_source, mapped_target) in gt_edge_set:
                        edge_tp += 1
                    elif (mapped_source is not None and mapped_source in gt_out) or (mapped_target is not None and mapped_target in gt_in):
                        edge_fp += 1

            components.at[idx, "visible_gt_matched_nodes"] = len(matched_node_ids)
            components.at[idx, "visible_gt_edge_tp"] = edge_tp
            components.at[idx, "visible_gt_edge_fp"] = edge_fp
            components.at[idx, "visible_gt_mean_match_um"] = float(np.mean(match_distances)) if match_distances else np.nan
            components.at[idx, "contains_visible_gt"] = int(bool(matched_node_ids or edge_tp or edge_fp))

    components.to_csv(SHORT7_COMPONENTS_PATH, index=False)
    print(f"[short7] annotated {len(components):,} components -> {SHORT7_COMPONENTS_PATH}")
    print(components[["contains_visible_gt", "visible_gt_matched_nodes", "visible_gt_edge_tp", "visible_gt_edge_fp"]].sum().to_string())


def main():
    if not SUBMISSION_PATH.exists():
        raise FileNotFoundError(f"submission.csv not found: {SUBMISSION_PATH}")
    df = pd.read_csv(SUBMISSION_PATH)
    datasets = sorted(set(df["dataset"].astype(str)))
    rows = []
    for dataset in datasets:
        gt_path = TRAIN_DIR / f"{dataset}.geff"
        if not gt_path.exists():
            print(f"[skip] {dataset}: no local GT at {gt_path}")
            continue
        pred_nodes, pred_edges = load_pred(df, dataset)
        row = eval_dataset(dataset, pred_nodes, pred_edges)
        rows.append(row)
        print(
            f"{dataset}: node_recall={row['node_recall']:.4f} "
            f"metric_edge_jaccard={row['metric_edge_jaccard_proxy']:.4f} "
            f"adjusted_edge_jaccard={row['adjusted_edge_jaccard_proxy']:.4f} "
            f"sparse_edge_recall={row['sparse_edge_recall']:.4f} "
            f"FP={row['edge_fp_metric']} FN={row['edge_fn_metric']} "
            f"mean_match_um={row['mean_match_um']:.3f}"
        )

    out = pd.DataFrame(rows)
    if not out.empty:
        total = {
            "dataset": "TOTAL",
            "pred_nodes": int(out["pred_nodes"].sum()),
            "gt_nodes": int(out["gt_nodes"].sum()),
            "node_tp": int(out["node_tp"].sum()),
            "pred_edges": int(out["pred_edges"].sum()),
            "gt_edges": int(out["gt_edges"].sum()),
            "edge_tp": int(out["edge_tp"].sum()),
            "edge_fp_metric": int(out["edge_fp_metric"].sum()),
            "edge_fn_metric": int(out["edge_fn_metric"].sum()),
            "ignored_pred_edges_metric": int(out["ignored_pred_edges_metric"].sum()),
            "estimated_true_nodes": int(out["estimated_true_nodes"].sum()),
            "pred_division_sources": int(out["pred_division_sources"].sum()),
            "gt_division_sources": int(out["gt_division_sources"].sum()),
        }
        total["node_precision"] = total["node_tp"] / total["pred_nodes"] if total["pred_nodes"] else 0.0
        total["node_recall"] = total["node_tp"] / total["gt_nodes"] if total["gt_nodes"] else 0.0
        total["node_f1"] = (
            2 * total["node_precision"] * total["node_recall"] / (total["node_precision"] + total["node_recall"])
            if (total["node_precision"] + total["node_recall"])
            else 0.0
        )
        total["strict_edge_precision_debug"] = total["edge_tp"] / total["pred_edges"] if total["pred_edges"] else 0.0
        total["sparse_edge_recall"] = total["edge_tp"] / total["gt_edges"] if total["gt_edges"] else 0.0
        total["metric_edge_jaccard_proxy"] = (
            total["edge_tp"] / (total["edge_tp"] + total["edge_fp_metric"] + total["edge_fn_metric"])
            if (total["edge_tp"] + total["edge_fp_metric"] + total["edge_fn_metric"])
            else 0.0
        )
        sample_weights = (
            out["edge_tp"].astype(float)
            + out["edge_fp_metric"].astype(float)
            + out["edge_fn_metric"].astype(float)
        )
        adjusted_weight_sum = float(sample_weights.sum())
        aggregate_node_penalty_debug = max(
            0.0,
            1.0
            - NODE_PENALTY_A
            * max(0, total["pred_nodes"] - total["estimated_true_nodes"])
            / max(1, total["estimated_true_nodes"]),
        )
        total["node_penalty_factor_proxy"] = (
            float(np.average(out["node_penalty_factor_proxy"].astype(float), weights=sample_weights))
            if adjusted_weight_sum > 0
            else 0.0
        )
        total["adjusted_edge_jaccard_proxy"] = (
            float(np.average(out["adjusted_edge_jaccard_proxy"].astype(float), weights=sample_weights))
            if adjusted_weight_sum > 0
            else 0.0
        )
        total["adjusted_edge_weight_sum"] = adjusted_weight_sum
        total["aggregate_node_penalty_debug"] = aggregate_node_penalty_debug
        total["aggregate_adjusted_edge_jaccard_debug"] = total["metric_edge_jaccard_proxy"] * aggregate_node_penalty_debug
        total["strict_edge_jaccard_debug"] = (
            total["edge_tp"] / (total["gt_edges"] + total["pred_edges"] - total["edge_tp"])
            if (total["gt_edges"] + total["pred_edges"] - total["edge_tp"])
            else 0.0
        )
        total["mean_match_um"] = np.nan
        total["p95_match_um"] = np.nan
        out = pd.concat([out, pd.DataFrame([total])], ignore_index=True)

    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(OUT_PATH, index=False)
    annotate_dense_short7_components(df)
    print(f"\nwrote {OUT_PATH}")
    try:
        from IPython.display import display

        display(out)
    except Exception:
        print(out.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
!python /kaggle/working/biohub_local_gt_eval.py